# Figure 2 and Supplementary Figures 6–8: multiplex LNP cell and functional analysis

This notebook combines the final published cell annotations with full-barcode LNP calls, applies the corrected LNP labels, defines OVA and SIINFEKL-H-2Kb expression gates from `S2_reg004` as the OVA-expression reference region, and produces the cell-level and region-level tables used in Figure 2a–e and j and Supplementary Figures 6–8. Cell segmentation and clustering follow the previously published workflow and are supplied as precomputed annotations. Manuscript plots display inline and are not written as image files.


## 1. Environment and publication paths


In [ ]:
from __future__ import annotations
from pathlib import Path
import math
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from IPython.display import display


In [ ]:
def find_nanostamp_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "Manuscripts" / "NanoSTAMP"
        if candidate.exists():
            return candidate
        if base.name == "NanoSTAMP" and (base / "Data").exists():
            return base
    raise FileNotFoundError(
        "Could not locate Manuscripts/NanoSTAMP. Start Jupyter inside the repository "
        "or set NANOSTAMP_ROOT to the publication package directory."
    )


NANOSTAMP_ROOT = find_nanostamp_root()
DATA_ROOT = NANOSTAMP_ROOT / "Data" / "Figure_2_and_Supplementary_Figures_6_11_Multiplex_LNP"
INPUT_ROOT = DATA_ROOT / "Precomputed_Downstream_Input"
GENERATED_ROOT = DATA_ROOT / "Generated_Output"

CELL_OUTPUT_ROOT = GENERATED_ROOT / "Cell_and_Functional_Analysis"
TABLE_DIR = CELL_OUTPUT_ROOT / "Tables"
FIGURE_DIR = CELL_OUTPUT_ROOT / "Inline_Figure_Placeholders"
OUTPUT_H5AD = CELL_OUTPUT_ROOT / "Figure_2_annotated_cells.h5ad"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

BATCH_INPUTS = {
    "S1_S2": {
        "annotation": INPUT_ROOT / "Round_1_S1_S2" / "20260628_10_LNP_subcluster_labeled_v3.h5ad",
        "spot_cell": INPUT_ROOT / "Round_1_S1_S2" / "cell_analysis_table_all_regions_v4.csv",
        "spot_summary": INPUT_ROOT / "Round_1_S1_S2" / "cell_analysis_summary_by_lnp_call_v4.csv",
    },
    "S3_S4": {
        "annotation": INPUT_ROOT / "Round_2_S3_S4" / "20260711_10_LNP_subcluster_labeled_v3.h5ad",
        "spot_cell": INPUT_ROOT / "Round_2_S3_S4" / "cell_analysis_table_all_regions_v4.csv",
        "spot_summary": INPUT_ROOT / "Round_2_S3_S4" / "cell_analysis_summary_by_lnp_call_v4.csv",
    },
}

REFERENCE_REGIONS = ["S2_reg004"]
LNP_REGIONS = [
    f"S{slide}_reg{region:03d}"
    for slide in range(1, 5)
    for region in range(4)
]
ANALYSIS_REGIONS = LNP_REGIONS + REFERENCE_REGIONS


LNP_ORDER = [f"LNP_{i:02d}" for i in range(1, 11)]
LNP_DETAIL_TABLE = pd.DataFrame([
    {"lnp_call": "LNP_01", "lnp_number": 1, "source_raw_lnp_call": "LNP_02", "ionizable_lipid": "SM-102", "helper_lipid": "DSPC", "formulation": "SM-102 + DSPC"},
    {"lnp_call": "LNP_02", "lnp_number": 2, "source_raw_lnp_call": "LNP_01", "ionizable_lipid": "SM-102", "helper_lipid": "DOTAP", "formulation": "SM-102 + DOTAP"},
    {"lnp_call": "LNP_03", "lnp_number": 3, "source_raw_lnp_call": "LNP_03", "ionizable_lipid": "SM-102", "helper_lipid": "DDAB", "formulation": "SM-102 + DDAB"},
    {"lnp_call": "LNP_04", "lnp_number": 4, "source_raw_lnp_call": "LNP_04", "ionizable_lipid": "SM-102", "helper_lipid": "DOPE", "formulation": "SM-102 + DOPE"},
    {"lnp_call": "LNP_05", "lnp_number": 5, "source_raw_lnp_call": "LNP_05", "ionizable_lipid": "SM-102", "helper_lipid": "18PG", "formulation": "SM-102 + 18PG"},
    {"lnp_call": "LNP_06", "lnp_number": 6, "source_raw_lnp_call": "LNP_06", "ionizable_lipid": "ALC-0315", "helper_lipid": "DSPC", "formulation": "ALC-0315 + DSPC"},
    {"lnp_call": "LNP_07", "lnp_number": 7, "source_raw_lnp_call": "LNP_07", "ionizable_lipid": "ALC-0315", "helper_lipid": "DOTAP", "formulation": "ALC-0315 + DOTAP"},
    {"lnp_call": "LNP_08", "lnp_number": 8, "source_raw_lnp_call": "LNP_08", "ionizable_lipid": "ALC-0315", "helper_lipid": "DDAB", "formulation": "ALC-0315 + DDAB"},
    {"lnp_call": "LNP_09", "lnp_number": 9, "source_raw_lnp_call": "LNP_09", "ionizable_lipid": "ALC-0315", "helper_lipid": "DOPE", "formulation": "ALC-0315 + DOPE"},
    {"lnp_call": "LNP_10", "lnp_number": 10, "source_raw_lnp_call": "LNP_10", "ionizable_lipid": "ALC-0315", "helper_lipid": "18PG", "formulation": "ALC-0315 + 18PG"},
])
RAW_TO_CORRECTED_LNP = dict(zip(
    LNP_DETAIL_TABLE["source_raw_lnp_call"],
    LNP_DETAIL_TABLE["lnp_call"],
))
LNP_TICK_LABELS = LNP_DETAIL_TABLE["lnp_call"].tolist()

for batch_paths in BATCH_INPUTS.values():
    for path in batch_paths.values():
        if not path.exists():
            raise FileNotFoundError(path)

print(f"Publication input root: {INPUT_ROOT}")
print(f"Generated tables:       {TABLE_DIR}")
print(f"Generated annotated data: {OUTPUT_H5AD}")


## 2. Plot style and statistical helpers


In [ ]:
FIGURE_DPI = 600
NATURE_COLORS = {'LNP': '#0F4D92', 'reference': '#D0D0D0', 'neutral': '#272727', 'grid': '#E6E6E6'}

def apply_nature_style(font_size: float=7, axes_linewidth: float=0.8) -> None:
    plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'figure.dpi': FIGURE_DPI, 'savefig.dpi': FIGURE_DPI, 'font.size': font_size, 'axes.spines.right': False, 'axes.spines.top': False, 'axes.linewidth': axes_linewidth, 'legend.frameon': False, 'xtick.major.width': axes_linewidth, 'ytick.major.width': axes_linewidth, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5})

def save_figure(fig: plt.Figure, stem: str, dpi: int=FIGURE_DPI) -> list[Path]:
    saved = []
    for suffix in ['svg', 'pdf', 'png']:
        path = FIGURE_DIR / f'{stem}.{suffix}'
        saved.append(path)
    return saved

def sem(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors='coerce').dropna()
    if len(values) <= 1:
        return 0.0
    return float(values.std(ddof=1) / math.sqrt(len(values)))
apply_nature_style()


## 3. Merge cell annotations with full-barcode LNP calls


In [ ]:
annotation_adatas = {batch: ad.read_h5ad(paths['annotation']) for (batch, paths) in BATCH_INPUTS.items()}
adata = ad.concat(annotation_adatas, join='outer', label='annotation_batch', index_unique='__', fill_value=np.nan, merge='same')
del annotation_adatas
spot = pd.concat([pd.read_csv(paths['spot_cell']).assign(spot_batch=batch) for (batch, paths) in BATCH_INPUTS.items()], ignore_index=True)
spot_summary = pd.concat([pd.read_csv(paths['spot_summary']).assign(spot_batch=batch) for (batch, paths) in BATCH_INPUTS.items()], ignore_index=True)
spot = spot[spot['region'].isin(ANALYSIS_REGIONS)].copy()
if 'region' in spot_summary.columns:
    spot_summary = spot_summary[spot_summary['region'].isin(ANALYSIS_REGIONS)].copy()
print(f'Annotation h5ad: {adata.n_obs:,} cells x {adata.n_vars:,} markers')
print(f'Spot per-cell table: {len(spot):,} rows')
print(f'Spot summary table: {len(spot_summary):,} rows')


In [ ]:
obs = adata.obs.copy()
obs['_obs_name'] = obs.index.astype(str)
obs['cell'] = obs['label'].astype(int)
obs['lnp_region'] = obs['slide_name'].astype(str).str.replace('^.*_registered_', '', regex=True) + '_' + obs['region'].astype(str)
keep_mask = obs['lnp_region'].isin(ANALYSIS_REGIONS)
print(f'Retaining {int(keep_mask.sum()):,} cells from the 16 analyzed LNP regions and the OVA-expression reference region.')
adata = adata[keep_mask.to_numpy()].copy()
obs = obs.loc[keep_mask].copy()
observed_regions = set(obs['lnp_region'].unique())
expected_regions = set(ANALYSIS_REGIONS)
if observed_regions != expected_regions:
    raise ValueError(f'Unexpected annotation regions: missing={sorted(expected_regions - observed_regions)}, extra={sorted(observed_regions - expected_regions)}')
marker_cols = ['DAPI', 'Podoplanin', 'aSMA', 'CD4', 'CD31', 'SCA1', 'CD152', 'CD45.2', 'Ly6C', 'PD1', 'CD3', 'CD11c', 'CD19', 'F480', 'CD138', 'CD169', 'CD62L', 'TCRb', 'GZMB', 'FOXP3', 'CD27', 'CD8a', 'CD11b', 'Fluc', 'MHCII', 'CCR7', 'CD86', 'CD28', 'B220', 'CD103', 'CD25', 'MPO', 'CD90', 'SIINFEKL_H-2Kb', 'CD44', 'OVA', 'NKp46']
lnp_marker_cols = ['A6', 'A17', 'A55', 'A56', 'A76', 'A79', 'A20', 'A46', 'A28', 'A72', 'A2', 'A63', 'A126', 'A127', 'A128']
spot_count_cols = [f'spot_{m}' for m in lnp_marker_cols]
bit_cols = [f'bit_{m}' for m in lnp_marker_cols]
spot_cols = ['region', 'cell', 'barcode', 'lnp_call', 'lnp_positive', 'barcode_in_library', 'barcode_match_distance', 'barcode_match_status', 'barcode_excluded', 'total_barcode_spots', 'decoded_exact_spots', 'decoded_tolerant_spots', 'n_positive_bits', 'dominant_decoded_barcode', 'dominant_decoded_barcode_count', 'dominant_barcode_marker', 'dominant_barcode_count', 'dominant_barcode_fraction', 'barcode_confidence', 'decoded_spots']
spot_cols += [c for c in marker_cols + lnp_marker_cols + spot_count_cols + bit_cols if c in spot.columns]
spot_merge = spot[spot_cols].rename(columns={'region': 'lnp_region'}).copy()
overlap_marker_cols = [c for c in marker_cols + lnp_marker_cols if c in obs.columns and c in spot_merge.columns]
spot_merge = spot_merge.rename(columns={c: f'spot_mean_{c}' for c in overlap_marker_cols})
merged_obs = obs.merge(spot_merge, on=['lnp_region', 'cell'], how='left', validate='one_to_one', indicator=True)
merge_counts = merged_obs['_merge'].value_counts(dropna=False)
extra_spot_rows = spot_merge.merge(obs[['lnp_region', 'cell']], on=['lnp_region', 'cell'], how='left', indicator=True)
extra_spot_rows = int((extra_spot_rows['_merge'] == 'left_only').sum())
print(merge_counts.to_string())
print(f'Extra spot-table cells without final annotation: {extra_spot_rows:,}')
if int((merged_obs['_merge'] != 'both').sum()) > 0:
    raise ValueError('Some annotated cells did not receive spot/LNP annotations.')
merged_obs = merged_obs.drop(columns=['_merge'])
merged_obs = merged_obs.set_index('_obs_name')
merged_obs.index.name = None
merged_obs = merged_obs.loc[adata.obs_names]


In [ ]:
merged_obs['condition'] = np.where(merged_obs['lnp_region'].isin(REFERENCE_REGIONS), 'reference', 'LNP')
merged_obs['region_group'] = pd.Categorical(merged_obs['lnp_region'], categories=ANALYSIS_REGIONS, ordered=True)
merged_obs['lnp_call'] = merged_obs['lnp_call'].fillna('no_barcode').astype(str)
merged_obs['lnp_call_raw'] = merged_obs['lnp_call']
merged_obs['lnp_call_corrected'] = merged_obs['lnp_call_raw'].map(RAW_TO_CORRECTED_LNP).fillna(merged_obs['lnp_call_raw'])
merged_obs['lnp_call'] = merged_obs['lnp_call_corrected']
merged_obs['lnp_positive'] = merged_obs['lnp_positive'].fillna(False).astype(bool)
merged_obs['lnp_call_positive_only'] = np.where(merged_obs['lnp_positive'], merged_obs['lnp_call_corrected'], 'no_barcode')
merged_obs = merged_obs.merge(LNP_DETAIL_TABLE.add_prefix('lnp_detail_'), left_on='lnp_call_corrected', right_on='lnp_detail_lnp_call', how='left')
merged_obs.index = adata.obs_names
LNP_DETAIL_TABLE.to_csv(TABLE_DIR / 'lnp_detail_table.csv', index=False)
display(LNP_DETAIL_TABLE)
adata.obs = merged_obs
adata.uns['spot_cell_annotation_merge'] = {'batch_inputs': {batch: {kind: str(path) for (kind, path) in paths.items()} for (batch, paths) in BATCH_INPUTS.items()}, 'output_h5ad': str(OUTPUT_H5AD), 'reference_regions': REFERENCE_REGIONS, 'lnp_regions': LNP_REGIONS, 'lnp_detail_table_json': LNP_DETAIL_TABLE.to_json(orient='records'), 'lnp_label_note': 'Display labels swap raw LNP_01 and raw LNP_02: corrected LNP_01 is raw LNP_02 (DSPC), corrected LNP_02 is raw LNP_01 (DOTAP).', 'extra_spot_rows_without_final_annotation': extra_spot_rows}
adata.write_h5ad(OUTPUT_H5AD)
print(f'Wrote {OUTPUT_H5AD}')


## 4. Per-region and cell-type-specific LNP uptake


In [ ]:
analysis_obs = adata.obs[['lnp_region', 'condition', 'cell_type', 'lnp_call_raw', 'lnp_call_corrected', 'lnp_positive']].copy()
analysis_obs = analysis_obs.rename(columns={'lnp_call_corrected': 'lnp_call'})
analysis_obs = analysis_obs[analysis_obs['lnp_region'].isin(LNP_REGIONS + REFERENCE_REGIONS)]
region_totals = analysis_obs.groupby(['condition', 'lnp_region'], observed=True).size().rename('n_cells_total').reset_index()
positive_counts = analysis_obs[analysis_obs['lnp_positive'] & analysis_obs['lnp_call'].isin(LNP_ORDER)].groupby(['condition', 'lnp_region', 'lnp_call'], observed=True).size().rename('n_lnp_positive').reset_index()
complete_index = pd.MultiIndex.from_product([['LNP', 'reference'], LNP_REGIONS + REFERENCE_REGIONS, LNP_ORDER], names=['condition', 'lnp_region', 'lnp_call']).to_frame(index=False)
complete_index = complete_index[(complete_index['condition'] == 'LNP') & complete_index['lnp_region'].isin(LNP_REGIONS) | (complete_index['condition'] == 'reference') & complete_index['lnp_region'].isin(REFERENCE_REGIONS)]
uptake_by_region = complete_index.merge(positive_counts, on=['condition', 'lnp_region', 'lnp_call'], how='left').merge(region_totals, on=['condition', 'lnp_region'], how='left')
uptake_by_region['n_lnp_positive'] = uptake_by_region['n_lnp_positive'].fillna(0).astype(int)
uptake_by_region['pct_cells_lnp_positive'] = 100 * uptake_by_region['n_lnp_positive'] / uptake_by_region['n_cells_total']
uptake_by_region['lnp_call'] = pd.Categorical(uptake_by_region['lnp_call'], categories=LNP_ORDER, ordered=True)
uptake_by_region = uptake_by_region.merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
uptake_by_region = uptake_by_region.sort_values(['condition', 'lnp_call', 'lnp_region'])
uptake_summary = uptake_by_region.groupby(['condition', 'lnp_call'], observed=True).agg(n_regions=('lnp_region', 'nunique'), mean_pct_cells_lnp_positive=('pct_cells_lnp_positive', 'mean'), sem_pct_cells_lnp_positive=('pct_cells_lnp_positive', sem), total_positive_cells=('n_lnp_positive', 'sum'), total_cells=('n_cells_total', 'sum')).reset_index().merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
uptake_summary['pooled_pct_cells_lnp_positive'] = 100 * uptake_summary['total_positive_cells'] / uptake_summary['total_cells']
uptake_by_region.to_csv(TABLE_DIR / 'lnp_uptake_percent_by_region.csv', index=False)
uptake_summary.to_csv(TABLE_DIR / 'lnp_uptake_percent_summary.csv', index=False)
display(uptake_summary)


In [ ]:
celltype_totals = analysis_obs[analysis_obs['condition'] == 'LNP'].groupby(['lnp_region', 'cell_type'], observed=True).size().rename('n_cell_type_total').reset_index()
celltype_positive = analysis_obs[(analysis_obs['condition'] == 'LNP') & analysis_obs['lnp_positive'] & analysis_obs['lnp_call'].isin(LNP_ORDER)].groupby(['lnp_region', 'cell_type', 'lnp_call'], observed=True).size().rename('n_lnp_positive').reset_index()
celltype_lnp_uptake = celltype_positive.merge(celltype_totals, on=['lnp_region', 'cell_type'], how='left')
celltype_lnp_uptake['pct_cell_type_lnp_positive'] = 100 * celltype_lnp_uptake['n_lnp_positive'] / celltype_lnp_uptake['n_cell_type_total']
celltype_lnp_summary = celltype_lnp_uptake.groupby(['cell_type', 'lnp_call'], observed=True).agg(n_regions=('lnp_region', 'nunique'), mean_pct_cell_type_lnp_positive=('pct_cell_type_lnp_positive', 'mean'), sem_pct_cell_type_lnp_positive=('pct_cell_type_lnp_positive', sem), total_positive_cells=('n_lnp_positive', 'sum'), median_region_pct=('pct_cell_type_lnp_positive', 'median')).reset_index().merge(LNP_DETAIL_TABLE, on='lnp_call', how='left').sort_values(['lnp_call', 'mean_pct_cell_type_lnp_positive'], ascending=[True, False])
celltype_lnp_uptake.to_csv(TABLE_DIR / 'lnp_uptake_by_cell_type_region.csv', index=False)
celltype_lnp_summary.to_csv(TABLE_DIR / 'lnp_uptake_by_cell_type_summary.csv', index=False)
display(celltype_lnp_summary.head(30))


In [ ]:
top_celltypes = celltype_lnp_summary.groupby('cell_type', observed=True)['total_positive_cells'].sum().sort_values(ascending=False).head(12).index
celltype_heatmap = celltype_lnp_summary[celltype_lnp_summary['cell_type'].isin(top_celltypes)].pivot(index='cell_type', columns='lnp_call', values='mean_pct_cell_type_lnp_positive').reindex(index=top_celltypes, columns=LNP_ORDER).fillna(0)
apply_nature_style()
(fig, ax) = plt.subplots(figsize=(5.4, 3.2))
sns.heatmap(celltype_heatmap, ax=ax, cmap='Blues', linewidths=0.35, linecolor='white', cbar_kws={'label': 'Mean % cell type LNP+', 'shrink': 0.75})
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('Top LNP+ cell types across LNP regions', fontsize=8, pad=5)
ax.text(-0.13, 1.04, 'd', transform=ax.transAxes, fontsize=8, fontweight='bold')
save_figure(fig, 'lnp_uptake_by_cell_type_heatmap')
plt.show()


### Supplementary Figure 6: cell-type-specific uptake


In [ ]:
apply_nature_style()
plot_celltypes_for_bars = list(top_celltypes)
complete_bar_index = pd.MultiIndex.from_product([LNP_REGIONS, plot_celltypes_for_bars, LNP_ORDER], names=['lnp_region', 'cell_type', 'lnp_call']).to_frame(index=False)
celltype_lnp_region_complete = complete_bar_index.merge(celltype_totals, on=['lnp_region', 'cell_type'], how='left').merge(celltype_positive, on=['lnp_region', 'cell_type', 'lnp_call'], how='left')
celltype_lnp_region_complete['n_cell_type_total'] = celltype_lnp_region_complete['n_cell_type_total'].fillna(0)
celltype_lnp_region_complete['n_lnp_positive'] = celltype_lnp_region_complete['n_lnp_positive'].fillna(0).astype(int)
celltype_lnp_region_complete['pct_cell_type_lnp_positive'] = np.where(celltype_lnp_region_complete['n_cell_type_total'] > 0, 100 * celltype_lnp_region_complete['n_lnp_positive'] / celltype_lnp_region_complete['n_cell_type_total'], np.nan)
celltype_bar_summary = celltype_lnp_region_complete.groupby(['cell_type', 'lnp_call'], observed=True).agg(mean_pct=('pct_cell_type_lnp_positive', 'mean'), sem_pct=('pct_cell_type_lnp_positive', sem)).reset_index()
n_cols = 3
n_rows = math.ceil(len(plot_celltypes_for_bars) / n_cols)
(fig, axes) = plt.subplots(n_rows, n_cols, figsize=(7.1, max(2.0, 1.75 * n_rows)), sharex=False, sharey=False)
axes = np.asarray(axes).ravel()
bar_color = NATURE_COLORS['LNP']
rng = np.random.default_rng(7)
x = np.arange(len(LNP_ORDER))
for (ax, cell_type) in zip(axes, plot_celltypes_for_bars):
    summary = celltype_bar_summary[celltype_bar_summary['cell_type'] == cell_type].set_index('lnp_call').reindex(LNP_ORDER)
    ax.bar(x, summary['mean_pct'], yerr=summary['sem_pct'], color=bar_color, edgecolor=NATURE_COLORS['neutral'], linewidth=0.55, width=0.72, error_kw={'elinewidth': 0.7, 'capthick': 0.7, 'capsize': 2.0}, zorder=2)
    points = celltype_lnp_region_complete[celltype_lnp_region_complete['cell_type'] == cell_type]
    for (i, lnp) in enumerate(LNP_ORDER):
        vals = points.loc[points['lnp_call'] == lnp, 'pct_cell_type_lnp_positive'].dropna().to_numpy()
        jitter = rng.uniform(-0.13, 0.13, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, s=8, facecolors='white', edgecolors=NATURE_COLORS['neutral'], linewidths=0.4, alpha=0.95, zorder=3)
    ymax_candidates = [summary['mean_pct'].add(summary['sem_pct'], fill_value=0).max()]
    if not points['pct_cell_type_lnp_positive'].dropna().empty:
        ymax_candidates.append(points['pct_cell_type_lnp_positive'].max())
    ymax = np.nanmax(ymax_candidates)
    ax.set_ylim(0, max(0.02, ymax * 1.22))
    ax.set_title(str(cell_type), fontsize=7, pad=4)
    ax.grid(axis='y', color=NATURE_COLORS['grid'], linewidth=0.45, zorder=0)
    ax.tick_params(axis='both', labelsize=6)
    ax.set_xticks(x)
    ax.set_xticklabels(LNP_TICK_LABELS, rotation=45, ha='right')
for ax in axes[len(plot_celltypes_for_bars):]:
    ax.axis('off')
for ax in axes[::n_cols]:
    ax.set_ylabel('% cell type LNP+', fontsize=7)
fig.suptitle('Per-cell-type LNP uptake profiles', fontsize=8, y=1.01)
fig.tight_layout(pad=0.8, h_pad=1.0, w_pad=0.8)
plt.show()


## 5. Cell-type composition of LNP-positive cells


In [ ]:
lnp_positive_cells = analysis_obs[(analysis_obs['condition'] == 'LNP') & analysis_obs['lnp_positive'] & analysis_obs['lnp_call'].isin(LNP_ORDER)].copy()
composition_counts = lnp_positive_cells.groupby(['lnp_call', 'cell_type'], observed=True).size().rename('n_lnp_positive').reset_index()
composition_celltypes = composition_counts.groupby('cell_type', observed=True)['n_lnp_positive'].sum().sort_values(ascending=False).head(10).index.tolist()
composition_complete = pd.MultiIndex.from_product([LNP_ORDER, composition_celltypes], names=['lnp_call', 'cell_type']).to_frame(index=False)
lnp_celltype_composition = composition_complete.merge(composition_counts, on=['lnp_call', 'cell_type'], how='left').merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
lnp_celltype_composition['n_lnp_positive'] = lnp_celltype_composition['n_lnp_positive'].fillna(0).astype(int)
lnp_celltype_composition['n_lnp_positive_total'] = lnp_celltype_composition.groupby('lnp_call', observed=True)['n_lnp_positive'].transform('sum')
lnp_celltype_composition['pct_of_lnp_positive_cells'] = np.where(lnp_celltype_composition['n_lnp_positive_total'] > 0, 100 * lnp_celltype_composition['n_lnp_positive'] / lnp_celltype_composition['n_lnp_positive_total'], np.nan)
lnp_celltype_composition.to_csv(TABLE_DIR / 'lnp_positive_cell_type_composition.csv', index=False)
composition_wide = lnp_celltype_composition.pivot(index='lnp_call', columns='cell_type', values='pct_of_lnp_positive_cells').reindex(index=LNP_ORDER, columns=composition_celltypes).fillna(0)
display(lnp_celltype_composition.head(20))


In [ ]:
baseline_celltype = analysis_obs[analysis_obs['condition'] == 'LNP'].groupby('cell_type', observed=True).size().rename('n_cells_background').reset_index()
baseline_celltype['background_fraction'] = baseline_celltype['n_cells_background'] / baseline_celltype['n_cells_background'].sum()
positive_celltype = analysis_obs[(analysis_obs['condition'] == 'LNP') & analysis_obs['lnp_positive'] & analysis_obs['lnp_call'].isin(LNP_ORDER)].groupby(['lnp_call', 'cell_type'], observed=True).size().rename('n_lnp_positive').reset_index()
complete_celltype_index = pd.MultiIndex.from_product([LNP_ORDER, baseline_celltype['cell_type'].tolist()], names=['lnp_call', 'cell_type']).to_frame(index=False)
relative_celltype_bias = complete_celltype_index.merge(positive_celltype, on=['lnp_call', 'cell_type'], how='left').merge(baseline_celltype, on='cell_type', how='left').merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
relative_celltype_bias['n_lnp_positive'] = relative_celltype_bias['n_lnp_positive'].fillna(0).astype(int)
relative_celltype_bias['n_lnp_positive_total'] = relative_celltype_bias.groupby('lnp_call', observed=True)['n_lnp_positive'].transform('sum')
relative_celltype_bias['positive_fraction_within_lnp'] = relative_celltype_bias['n_lnp_positive'] / relative_celltype_bias['n_lnp_positive_total'].replace(0, np.nan)
relative_celltype_bias['enrichment_vs_background'] = relative_celltype_bias['positive_fraction_within_lnp'] / relative_celltype_bias['background_fraction'].replace(0, np.nan)
relative_celltype_bias['log2_enrichment_vs_background'] = np.log2(relative_celltype_bias['enrichment_vs_background'].replace(0, np.nan))
celltype_lnp_relative = relative_celltype_bias.copy()
celltype_lnp_relative['max_enrichment_for_cell_type'] = celltype_lnp_relative.groupby('cell_type', observed=True)['enrichment_vs_background'].transform('max')
celltype_lnp_relative['relative_enrichment_within_cell_type'] = celltype_lnp_relative['enrichment_vs_background'] / celltype_lnp_relative['max_enrichment_for_cell_type'].replace(0, np.nan)
best_lnp_by_celltype = celltype_lnp_relative.sort_values(['cell_type', 'enrichment_vs_background'], ascending=[True, False]).groupby('cell_type', observed=True).head(1)[['cell_type', 'lnp_call', 'formulation', 'helper_lipid', 'ionizable_lipid', 'n_lnp_positive', 'positive_fraction_within_lnp', 'background_fraction', 'enrichment_vs_background', 'log2_enrichment_vs_background']].sort_values('enrichment_vs_background', ascending=False)
relative_celltype_bias.to_csv(TABLE_DIR / 'lnp_cell_type_relative_enrichment.csv', index=False)
celltype_lnp_relative.to_csv(TABLE_DIR / 'lnp_cell_type_relative_enrichment_row_normalized.csv', index=False)
best_lnp_by_celltype.to_csv(TABLE_DIR / 'best_lnp_by_cell_type_relative_enrichment.csv', index=False)
display(best_lnp_by_celltype.head(20))


## 6. OVA-expression gating and functional summaries


In [ ]:
GENE_MARKERS = {'Luc': 'Fluc', 'OVA': 'OVA'}
missing_gene_markers = [col for col in GENE_MARKERS.values() if col not in adata.obs.columns]
if missing_gene_markers:
    raise KeyError(f'Missing expression columns in adata.obs: {missing_gene_markers}')
REFERENCE_GATE_QUANTILE = 0.95
GENE_GATE_REFERENCE_REGIONS = ['S2_reg004']

def reference_quantile_gate(values: pd.Series, quantile: float=REFERENCE_GATE_QUANTILE) -> float:
    values = pd.to_numeric(values, errors='coerce').dropna()
    if values.empty:
        return np.nan
    return float(values.quantile(quantile))
expression_cols = ['lnp_region', 'condition', 'cell_type', 'lnp_call', 'lnp_call_raw', 'lnp_positive', 'lnp_detail_formulation', 'lnp_detail_helper_lipid'] + list(GENE_MARKERS.values())
expression_obs = adata.obs[expression_cols].copy()
gate_rows = []
for (marker_name, col) in GENE_MARKERS.items():
    reference_values = expression_obs.loc[expression_obs['lnp_region'].isin(GENE_GATE_REFERENCE_REGIONS), col]
    threshold = reference_quantile_gate(reference_values)
    expression_obs[f'{marker_name}_positive'] = expression_obs[col] > threshold
    gate_rows.append({'marker': marker_name, 'column': col, 'gate_reference_regions': ','.join(GENE_GATE_REFERENCE_REGIONS), 'threshold': threshold, 'reference_median': pd.to_numeric(reference_values, errors='coerce').median(), 'reference_gate_quantile': REFERENCE_GATE_QUANTILE, 'reference_gate_value': pd.to_numeric(reference_values, errors='coerce').quantile(REFERENCE_GATE_QUANTILE), 'reference_n_cells': int(pd.to_numeric(reference_values, errors='coerce').notna().sum())})
expression_obs['Luc_OVA_double_positive'] = expression_obs['Luc_positive'] & expression_obs['OVA_positive']
gene_gate_table = pd.DataFrame(gate_rows)
gene_gate_table.to_csv(TABLE_DIR / 'luc_ova_reference_gate_thresholds.csv', index=False)
display(gene_gate_table)


In [ ]:
def summarize_gene_group(df: pd.DataFrame, group_cols: list[str], label: str) -> pd.DataFrame:
    summary = df.groupby(group_cols, observed=True).agg(n_cells=('condition', 'size'), n_lnp_positive=('lnp_positive', 'sum'), n_luc_positive=('Luc_positive', 'sum'), n_ova_positive=('OVA_positive', 'sum'), n_luc_ova_double_positive=('Luc_OVA_double_positive', 'sum'), mean_luc=('Fluc', 'mean'), mean_ova=('OVA', 'mean'), median_luc=('Fluc', 'median'), median_ova=('OVA', 'median')).reset_index()
    summary['analysis_population'] = label
    summary['pct_luc_positive'] = 100 * summary['n_luc_positive'] / summary['n_cells'].replace(0, np.nan)
    summary['pct_ova_positive'] = 100 * summary['n_ova_positive'] / summary['n_cells'].replace(0, np.nan)
    summary['pct_luc_ova_double_positive'] = 100 * summary['n_luc_ova_double_positive'] / summary['n_cells'].replace(0, np.nan)
    return summary
gene_region_all_cells = summarize_gene_group(expression_obs, ['condition', 'lnp_region', 'lnp_call'], 'all_cells')
gene_region_lnp_positive = summarize_gene_group(expression_obs[expression_obs['lnp_positive'] & expression_obs['lnp_call'].isin(LNP_ORDER)], ['condition', 'lnp_region', 'lnp_call'], 'lnp_positive_cells')
gene_region_summary = pd.concat([gene_region_all_cells, gene_region_lnp_positive], ignore_index=True)
gene_region_summary = gene_region_summary.merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
gene_region_summary.to_csv(TABLE_DIR / 'luc_ova_expression_by_region.csv', index=False)
gene_lnp_positive_summary = gene_region_lnp_positive[gene_region_lnp_positive['condition'] == 'LNP'].groupby('lnp_call', observed=True).agg(n_regions=('lnp_region', 'nunique'), mean_pct_luc_positive=('pct_luc_positive', 'mean'), sem_pct_luc_positive=('pct_luc_positive', sem), mean_pct_ova_positive=('pct_ova_positive', 'mean'), sem_pct_ova_positive=('pct_ova_positive', sem), mean_pct_luc_ova_double_positive=('pct_luc_ova_double_positive', 'mean'), sem_pct_luc_ova_double_positive=('pct_luc_ova_double_positive', sem), total_lnp_positive_cells=('n_cells', 'sum'), total_luc_positive=('n_luc_positive', 'sum'), total_ova_positive=('n_ova_positive', 'sum')).reset_index().merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
gene_lnp_positive_summary.to_csv(TABLE_DIR / 'luc_ova_expression_by_lnp_positive_cells.csv', index=False)
display(gene_lnp_positive_summary)


In [ ]:
apply_nature_style()
pooled_gene_expr = expression_obs[expression_obs['condition'] == 'LNP'].copy()
pooled_gene_expr['lnp_label_for_gene_positive'] = np.where(pooled_gene_expr['lnp_positive'].fillna(False).astype(bool) & pooled_gene_expr['lnp_call'].isin(LNP_ORDER), pooled_gene_expr['lnp_call'].astype(str), 'no_lnp_signal')
pooled_gene_flags = {'Luc': 'Luc_positive', 'OVA': 'OVA_positive'}
lnp_label_order_for_pies = LNP_ORDER + ['no_lnp_signal']
lnp_label_colors = dict(zip(LNP_ORDER, sns.color_palette('tab20', n_colors=len(LNP_ORDER))))
lnp_label_colors['no_lnp_signal'] = '#BDBDBD'
composition_rows = []
summary_rows = []
for (marker_name, flag_col) in pooled_gene_flags.items():
    marker_positive = pooled_gene_expr[pooled_gene_expr[flag_col]].copy()
    counts = marker_positive['lnp_label_for_gene_positive'].value_counts().reindex(lnp_label_order_for_pies, fill_value=0).astype(int)
    n_total = int(counts.sum())
    n_lnp_labeled = int(counts.reindex(LNP_ORDER, fill_value=0).sum())
    pct_lnp_labeled = 100 * n_lnp_labeled / n_total if n_total else np.nan
    for (label, count) in counts.items():
        composition_rows.append({'marker': marker_name, 'lnp_label': label, 'n_positive_cells': int(count), 'pct_of_marker_positive_cells': 100 * count / n_total if n_total else np.nan})
    summary_rows.append({'marker': marker_name, 'n_positive_cells': n_total, 'n_lnp_labeled': n_lnp_labeled, 'pct_lnp_labeled': pct_lnp_labeled, 'n_no_lnp_signal': int(counts['no_lnp_signal'])})
pooled_gene_positive_lnp_composition = pd.DataFrame(composition_rows)
pooled_gene_positive_lnp_summary = pd.DataFrame(summary_rows)
pooled_gene_positive_lnp_composition.to_csv(TABLE_DIR / 'luc_ova_positive_all_cells_lnp_label_composition.csv', index=False)
pooled_gene_positive_lnp_summary.to_csv(TABLE_DIR / 'luc_ova_positive_all_cells_lnp_labeled_summary.csv', index=False)
(fig, axes) = plt.subplots(1, 2, figsize=(4.6, 2.2), subplot_kw={'aspect': 'equal'})
for (ax, marker_name) in zip(axes, pooled_gene_flags):
    plot_data = pooled_gene_positive_lnp_composition[pooled_gene_positive_lnp_composition['marker'] == marker_name].set_index('lnp_label').reindex(lnp_label_order_for_pies)
    vals = plot_data['n_positive_cells'].fillna(0).to_numpy(dtype=float)
    nonzero = vals > 0
    labels = [label for (label, keep) in zip(lnp_label_order_for_pies, nonzero) if keep]
    colors = [lnp_label_colors[label] for label in labels]
    if vals[nonzero].sum() > 0:
        ax.pie(vals[nonzero], colors=colors, startangle=90, counterclock=False, wedgeprops={'linewidth': 0.25, 'edgecolor': 'white'})
    marker_summary = pooled_gene_positive_lnp_summary.set_index('marker').loc[marker_name]
    ax.set_title(f"All {marker_name}+ cells{chr(10)}n={int(marker_summary['n_positive_cells']):,}; LNP-labeled={marker_summary['pct_lnp_labeled']:.1f}%", fontsize=7, pad=3)
fig.legend(handles=[plt.Line2D([0], [0], marker='s', linestyle='', markersize=5, markerfacecolor=lnp_label_colors[label], markeredgecolor='none', label=str(label)) for label in lnp_label_order_for_pies], title='LNP label', fontsize=5.2, title_fontsize=6, bbox_to_anchor=(1.01, 0.5), loc='center left', borderaxespad=0)
fig.suptitle('LNP labels among all Luc+ and OVA+ cells in LNP-treated regions', fontsize=8, y=1.03)
fig.tight_layout(pad=0.6, w_pad=0.5)
plt.show()
display(pooled_gene_positive_lnp_summary)


In [ ]:
gene_expr_lnp_regions = expression_obs[expression_obs['condition'] == 'LNP'].copy()
gene_expr_lnp_regions['lnp_label_for_gene_positive'] = np.where(gene_expr_lnp_regions['lnp_positive'] & gene_expr_lnp_regions['lnp_call'].isin(LNP_ORDER), gene_expr_lnp_regions['lnp_call'], 'no_lnp_signal')
GENE_POSITIVE_FLAGS = {'Luc': 'Luc_positive', 'OVA': 'OVA_positive'}
SELECTED_GENE_CELLTYPES = ['DC', 'CD8+ T', 'CD4+ T', 'B', 'Macrophage']
selected_gene_celltypes_present = [ct for ct in SELECTED_GENE_CELLTYPES if ct in set(gene_expr_lnp_regions['cell_type'].dropna())]
celltype_comp_rows = []
lnp_label_comp_rows = []
for (marker_name, flag_col) in GENE_POSITIVE_FLAGS.items():
    marker_positive = gene_expr_lnp_regions[gene_expr_lnp_regions[flag_col]].copy()
    marker_total = len(marker_positive)
    celltype_counts = marker_positive.groupby('cell_type', observed=True).size().rename('n_positive_cells').reset_index()
    celltype_counts['marker'] = marker_name
    celltype_counts['total_positive_cells'] = marker_total
    celltype_counts['pct_positive_cells'] = 100 * celltype_counts['n_positive_cells'] / max(marker_total, 1)
    celltype_comp_rows.append(celltype_counts)
    selected_positive = marker_positive[marker_positive['cell_type'].isin(selected_gene_celltypes_present)].copy()
    lnp_counts = selected_positive.groupby(['cell_type', 'lnp_label_for_gene_positive'], observed=True).size().rename('n_positive_cells').reset_index()
    complete_lnp_index = pd.MultiIndex.from_product([selected_gene_celltypes_present, LNP_ORDER + ['no_lnp_signal']], names=['cell_type', 'lnp_label_for_gene_positive']).to_frame(index=False)
    lnp_counts = complete_lnp_index.merge(lnp_counts, on=['cell_type', 'lnp_label_for_gene_positive'], how='left')
    lnp_counts['n_positive_cells'] = lnp_counts['n_positive_cells'].fillna(0).astype(int)
    lnp_counts['marker'] = marker_name
    lnp_counts['total_positive_cells_in_cell_type'] = lnp_counts.groupby('cell_type', observed=True)['n_positive_cells'].transform('sum')
    lnp_counts['pct_positive_cells_in_cell_type'] = np.where(lnp_counts['total_positive_cells_in_cell_type'] > 0, 100 * lnp_counts['n_positive_cells'] / lnp_counts['total_positive_cells_in_cell_type'], np.nan)
    lnp_label_comp_rows.append(lnp_counts)
gene_positive_celltype_composition = pd.concat(celltype_comp_rows, ignore_index=True)
gene_positive_lnp_label_composition = pd.concat(lnp_label_comp_rows, ignore_index=True)
gene_positive_celltype_composition.to_csv(TABLE_DIR / 'luc_ova_positive_cell_type_composition.csv', index=False)
gene_positive_lnp_label_composition.to_csv(TABLE_DIR / 'luc_ova_positive_selected_cell_type_lnp_label_composition.csv', index=False)
display(gene_positive_celltype_composition.sort_values(['marker', 'n_positive_cells'], ascending=[True, False]).head(30))
display(gene_positive_lnp_label_composition.head(30))


### Supplementary Figure 8: OVA expression among LNP-positive cell types


In [ ]:
from matplotlib.colors import PowerNorm
apply_nature_style()
GROUPED_CELL_TYPE_ORDER = ['B', 'T', 'Macrophage', 'Non-immune', 'DC']
GROUPED_CELL_TYPE_MAP = {'B': 'B', 'CD4+ T': 'T', 'CD8+ T': 'T', 'Macrophage': 'Macrophage', 'DC': 'DC', 'Endothelial': 'Non-immune', 'Fibroblast': 'Non-immune', 'Lymphatic Endothelial': 'Non-immune', 'Muscle': 'Non-immune', 'Stromal': 'Non-immune'}
lnp_positive_grouped_cells = expression_obs[(expression_obs['condition'] == 'LNP') & expression_obs['lnp_positive'].fillna(False).astype(bool) & expression_obs['lnp_call'].isin(LNP_ORDER)].copy()
lnp_positive_grouped_cells['cell_type_group'] = lnp_positive_grouped_cells['cell_type'].astype(str).map(GROUPED_CELL_TYPE_MAP)
lnp_positive_grouped_cells = lnp_positive_grouped_cells[lnp_positive_grouped_cells['cell_type_group'].notna()].copy()
luc_ova_grouped_lnp = lnp_positive_grouped_cells.groupby(['cell_type_group', 'lnp_call'], observed=True).agg(n_lnp_positive_cells=('condition', 'size'), n_luc_positive=('Luc_positive', 'sum'), n_ova_positive=('OVA_positive', 'sum')).reset_index()
complete_grouped_lnp = pd.MultiIndex.from_product([GROUPED_CELL_TYPE_ORDER, LNP_ORDER], names=['cell_type_group', 'lnp_call']).to_frame(index=False)
luc_ova_grouped_lnp = complete_grouped_lnp.merge(luc_ova_grouped_lnp, on=['cell_type_group', 'lnp_call'], how='left')
for count_col in ['n_lnp_positive_cells', 'n_luc_positive', 'n_ova_positive']:
    luc_ova_grouped_lnp[count_col] = luc_ova_grouped_lnp[count_col].fillna(0).astype(int)
luc_ova_grouped_lnp['pct_luc_positive'] = 100 * luc_ova_grouped_lnp['n_luc_positive'] / luc_ova_grouped_lnp['n_lnp_positive_cells'].replace(0, np.nan)
luc_ova_grouped_lnp['pct_ova_positive'] = 100 * luc_ova_grouped_lnp['n_ova_positive'] / luc_ova_grouped_lnp['n_lnp_positive_cells'].replace(0, np.nan)
luc_ova_grouped_lnp = luc_ova_grouped_lnp.merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
luc_ova_grouped_lnp.to_csv(TABLE_DIR / 'luc_ova_rate_within_lnp_positive_cells_grouped_cell_types.csv', index=False)
heatmap_specs = [('pct_luc_positive', 'Luc+ rate within LNP+ cells'), ('pct_ova_positive', 'OVA+ rate within LNP+ cells')]
heatmap_matrices = []
for (metric, _) in heatmap_specs:
    matrix = luc_ova_grouped_lnp.pivot(index='cell_type_group', columns='lnp_call', values=metric).reindex(index=GROUPED_CELL_TYPE_ORDER, columns=LNP_ORDER)
    heatmap_matrices.append(matrix)
finite_values = np.concatenate([matrix.to_numpy(dtype=float)[np.isfinite(matrix.to_numpy(dtype=float))] for matrix in heatmap_matrices])
shared_vmax = max(float(finite_values.max()), 1.0) if len(finite_values) else 1.0
colorbar_max = 5 * np.ceil(shared_vmax / 5)
color_norm = PowerNorm(gamma=0.65, vmin=0, vmax=colorbar_max)
colorbar_ticks = np.arange(0, colorbar_max + 0.1, 5)
(fig, axes) = plt.subplots(1, 2, figsize=(7.1, 2.8), sharey=True)
for (panel_index, (ax, matrix, (_, title))) in enumerate(zip(axes, heatmap_matrices, heatmap_specs)):
    sns.heatmap(matrix, ax=ax, cmap='magma_r', norm=color_norm, linewidths=0.5, linecolor='white', mask=matrix.isna(), cbar=panel_index == 1, cbar_kws={'label': 'Positive among decoded LNP+ cells (%)', 'shrink': 0.78, 'ticks': colorbar_ticks})
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix.iat[row_index, column_index]
            if pd.notna(value):
                text_color = 'white' if color_norm(value) >= 0.58 else NATURE_COLORS['neutral']
                ax.text(column_index + 0.5, row_index + 0.5, f'{value:.1f}%', ha='center', va='center', fontsize=5.2, color=text_color)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title, fontsize=8, pad=5)
    ax.set_xticklabels(LNP_TICK_LABELS, rotation=45, ha='right')
    ax.set_yticklabels(GROUPED_CELL_TYPE_ORDER, rotation=0)
fig.suptitle('Luc/OVA+ rates within decoded LNP+ cells by grouped cell type', fontsize=9, y=1.02)
fig.tight_layout(pad=0.8, w_pad=0.8)
save_figure(fig, 'luc_ova_rate_within_lnp_positive_cells_grouped_celltype_heatmaps')
plt.show()
display(luc_ova_grouped_lnp[['cell_type_group', 'lnp_call', 'n_lnp_positive_cells', 'n_luc_positive', 'pct_luc_positive', 'n_ova_positive', 'pct_ova_positive']])


## 7. SIINFEKL-H-2Kb-positive LNP-positive dendritic cells


In [ ]:
SIINFEKL_COLUMN = 'SIINFEKL_H-2Kb'
CD86_COLUMN = 'CD86'
SIINFEKL_GATE_QUANTILE = 0.9
missing_dc_expression_cols = [c for c in [SIINFEKL_COLUMN, CD86_COLUMN] if c not in adata.obs.columns]
if missing_dc_expression_cols:
    raise KeyError(f'Missing expression columns in adata.obs: {missing_dc_expression_cols}')
expression_obs[SIINFEKL_COLUMN] = pd.to_numeric(adata.obs.loc[expression_obs.index, SIINFEKL_COLUMN], errors='coerce')
expression_obs[CD86_COLUMN] = pd.to_numeric(adata.obs.loc[expression_obs.index, CD86_COLUMN], errors='coerce')
siin_reference_values = expression_obs.loc[expression_obs['lnp_region'].isin(GENE_GATE_REFERENCE_REGIONS), SIINFEKL_COLUMN].dropna()
SIINFEKL_THRESHOLD = reference_quantile_gate(siin_reference_values, quantile=SIINFEKL_GATE_QUANTILE)
expression_obs['SIINFEKL_positive'] = expression_obs[SIINFEKL_COLUMN] > SIINFEKL_THRESHOLD
siin_gate_table = pd.DataFrame([{'marker': 'SIINFEKL-H-2Kb', 'column': SIINFEKL_COLUMN, 'threshold': SIINFEKL_THRESHOLD, 'gate_quantile': SIINFEKL_GATE_QUANTILE, 'gate_reference_regions': ','.join(GENE_GATE_REFERENCE_REGIONS), 'reference_median': siin_reference_values.median(), 'reference_n_cells': int(siin_reference_values.notna().sum())}])
siin_gate_table.to_csv(TABLE_DIR / 'siinfekl_h2kb_s2_reg004_gate_threshold.csv', index=False)
display(siin_gate_table)
dc_lnp_positive = expression_obs[(expression_obs['condition'] == 'LNP') & (expression_obs['cell_type'].astype(str) == 'DC') & expression_obs['lnp_positive'] & expression_obs['lnp_call'].isin(LNP_ORDER)].copy()
dc_lnp_positive['CD86_in_SIINFEKL_positive'] = dc_lnp_positive[CD86_COLUMN].where(dc_lnp_positive['SIINFEKL_positive'])
dc_siin_ova_by_region = dc_lnp_positive.groupby(['lnp_region', 'lnp_call'], observed=True).agg(n_lnp_positive_dc=('lnp_call', 'size'), n_siinfekl_positive=('SIINFEKL_positive', 'sum'), n_ova_positive=('OVA_positive', 'sum'), mean_siinfekl_expression=(SIINFEKL_COLUMN, 'mean'), median_siinfekl_expression=(SIINFEKL_COLUMN, 'median'), mean_cd86_in_siinfekl_positive=('CD86_in_SIINFEKL_positive', 'mean'), median_cd86_in_siinfekl_positive=('CD86_in_SIINFEKL_positive', 'median')).reset_index()
dc_siin_ova_by_region['pct_siinfekl_positive'] = 100 * dc_siin_ova_by_region['n_siinfekl_positive'] / dc_siin_ova_by_region['n_lnp_positive_dc']
dc_siin_ova_by_region['pct_ova_positive'] = 100 * dc_siin_ova_by_region['n_ova_positive'] / dc_siin_ova_by_region['n_lnp_positive_dc']
dc_siin_ova_by_region['siinfekl_minus_ova_pct_points'] = dc_siin_ova_by_region['pct_siinfekl_positive'] - dc_siin_ova_by_region['pct_ova_positive']
dc_siin_ova_summary = dc_siin_ova_by_region.groupby('lnp_call', observed=True).agg(n_regions=('lnp_region', 'nunique'), total_lnp_positive_dc=('n_lnp_positive_dc', 'sum'), total_siinfekl_positive=('n_siinfekl_positive', 'sum'), total_ova_positive=('n_ova_positive', 'sum'), mean_pct_siinfekl_positive=('pct_siinfekl_positive', 'mean'), sem_pct_siinfekl_positive=('pct_siinfekl_positive', sem), mean_pct_ova_positive=('pct_ova_positive', 'mean'), sem_pct_ova_positive=('pct_ova_positive', sem), mean_siinfekl_expression=('mean_siinfekl_expression', 'mean'), sem_siinfekl_expression=('mean_siinfekl_expression', sem), mean_cd86_in_siinfekl_positive=('mean_cd86_in_siinfekl_positive', 'mean'), sem_cd86_in_siinfekl_positive=('mean_cd86_in_siinfekl_positive', sem)).reindex(LNP_ORDER).reset_index()
dc_siin_ova_summary['pooled_pct_siinfekl_positive'] = 100 * dc_siin_ova_summary['total_siinfekl_positive'] / dc_siin_ova_summary['total_lnp_positive_dc']
dc_siin_ova_summary['pooled_pct_ova_positive'] = 100 * dc_siin_ova_summary['total_ova_positive'] / dc_siin_ova_summary['total_lnp_positive_dc']
dc_siin_ova_summary['mean_siinfekl_minus_ova_pct_points'] = dc_siin_ova_summary['mean_pct_siinfekl_positive'] - dc_siin_ova_summary['mean_pct_ova_positive']
dc_siin_ova_summary = dc_siin_ova_summary.merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
dc_siin_ova_by_region.to_csv(TABLE_DIR / 'lnp_positive_dc_siinfekl_ova_by_region.csv', index=False)
dc_siin_ova_summary.to_csv(TABLE_DIR / 'lnp_positive_dc_siinfekl_ova_summary.csv', index=False)
display(dc_siin_ova_summary)


In [ ]:
siinfekl_positive_lnp_dc = dc_lnp_positive[dc_lnp_positive['SIINFEKL_positive']].copy()
siinfekl_lnp_composition = siinfekl_positive_lnp_dc.groupby('lnp_call', observed=True).size().reindex(LNP_ORDER, fill_value=0).rename('n_siinfekl_positive_lnp_dc').reset_index()
siinfekl_lnp_composition['pct_siinfekl_positive_lnp_dc'] = 100 * siinfekl_lnp_composition['n_siinfekl_positive_lnp_dc'] / max(int(siinfekl_lnp_composition['n_siinfekl_positive_lnp_dc'].sum()), 1)
siinfekl_lnp_composition = siinfekl_lnp_composition.merge(LNP_DETAIL_TABLE, on='lnp_call', how='left')
siinfekl_lnp_composition.to_csv(TABLE_DIR / 'siinfekl_positive_lnp_dc_lnp_composition.csv', index=False)
display(siinfekl_lnp_composition)
apply_nature_style()
pie_values = siinfekl_lnp_composition.set_index('lnp_call').reindex(LNP_ORDER)['n_siinfekl_positive_lnp_dc'].to_numpy(dtype=float)
pie_colors = sns.color_palette('tab20', n_colors=len(LNP_ORDER))
(fig, ax) = plt.subplots(figsize=(4.4, 3.5), subplot_kw={'aspect': 'equal'})
if pie_values.sum() > 0:
    (wedges, _, _) = ax.pie(pie_values, colors=pie_colors, startangle=90, counterclock=False, autopct=lambda pct: f'{pct:.1f}%' if pct >= 2 else '', pctdistance=0.72, textprops={'fontsize': 6}, wedgeprops={'linewidth': 0.4, 'edgecolor': 'white'})
    legend_labels = [f'{lnp} (n={int(n):,})' for (lnp, n) in zip(LNP_ORDER, pie_values)]
    ax.legend(wedges, legend_labels, title='LNP label', fontsize=6, title_fontsize=7, bbox_to_anchor=(1.02, 0.5), loc='center left', borderaxespad=0)
total_siinfekl_lnp_dc = int(pie_values.sum())
ax.set_title(f'LNP composition of SIINFEKL-H-2Kb+ LNP+ DCs\nn={total_siinfekl_lnp_dc:,}', fontsize=8, pad=6)
fig.tight_layout(pad=0.8)
save_figure(fig, 'siinfekl_positive_lnp_dc_lnp_composition_pie')
plt.show()


## 8. Region-level quality-control tables


In [ ]:
region_qc = analysis_obs.groupby(['condition', 'lnp_region'], observed=True).agg(n_cells=('lnp_call', 'size'), n_any_lnp_positive=('lnp_positive', 'sum')).reset_index()
region_qc['pct_any_lnp_positive'] = 100 * region_qc['n_any_lnp_positive'] / region_qc['n_cells']
call_qc = analysis_obs.groupby(['condition', 'lnp_call'], observed=True).agg(n_cells=('lnp_call', 'size'), n_positive=('lnp_positive', 'sum')).reset_index().merge(LNP_DETAIL_TABLE, on='lnp_call', how='left').sort_values(['condition', 'n_cells'], ascending=[True, False])
region_qc.to_csv(TABLE_DIR / 'region_lnp_positive_qc.csv', index=False)
call_qc.to_csv(TABLE_DIR / 'lnp_call_qc.csv', index=False)
display(region_qc)
display(call_qc.head(30))


## 9. Figure 2b: selected-region spatial mapping


In [ ]:
from pathlib import Path
import re
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
CATPLOT_PROJECT_DIR = NANOSTAMP_ROOT
CATPLOT_RESULTS_DIR = CATPLOT_PROJECT_DIR / 'output/20260708_multiplex_10_LNP_R2/registration_results'
CATPLOT_FIGURE_DIR = CATPLOT_RESULTS_DIR / '20260712_spot_cell_annotation_downstream_S1_S4_figures'
CATPLOT_TABLE_DIR = CATPLOT_RESULTS_DIR / '20260712_spot_cell_annotation_downstream_S1_S4_tables'
CATPLOT_OUTPUT_H5AD = OUTPUT_H5AD
CATPLOT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CATPLOT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
CATPLOT_LNP_ORDER = [f'LNP_{i:02d}' for i in range(1, 11)]
CATPLOT_LNP_REGIONS = [f'S{slide}_reg{region:03d}' for slide in range(1, 5) for region in range(4)]
CATPLOT_REGIONS_TO_PLOT = None
CATPLOT_BACKGROUND_MAX_CELLS_PER_REGION = 120000
CATPLOT_RANDOM_SEED = 7
CATPLOT_DPI = 600
LNP_PALETTE = {'LNP_01': '#2C7FB8', 'LNP_02': '#A6C8E6', 'LNP_03': '#F28E2B', 'LNP_04': '#FDBF6F', 'LNP_05': '#2CA25F', 'LNP_06': '#99D594', 'LNP_07': '#D62728', 'LNP_08': '#FB8C8C', 'LNP_09': '#9467BD', 'LNP_10': '#C7B3D8'}
CELL_TYPE_PALETTE = {'B': '#4C78A8', 'Macrophage': '#59A14F', 'Endothelial': '#F28E2B', 'CD4+ T': '#9ECAE1', 'CD8+ T': '#6BAED6', 'DC': '#2CA02C', 'Fibroblast': '#D62728', 'Muscle': '#FF9896', 'Neutrophil': '#9467BD', 'Stromal': '#C5B0D5', 'Other': '#B9B0AC', 'Unknown': '#8E8E8E'}

def apply_catplot_nature_style():
    mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.titlesize': 8, 'axes.labelsize': 7, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'savefig.dpi': CATPLOT_DPI, 'savefig.bbox': 'tight'})
    sns.set_style('white')

def safe_name(value):
    return re.sub('[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

def catplot2(df, hue, exp='Exp', X='X', Y='Y', invert_y=False, size=3, legend=True, palette='bright', figsize=(8, 6), style='white', exps=None, axis='on', scatter_kws=None, save_path=None, dpi=600, formats=('pdf', 'png')):
    """Template-style categorical spatial plot, with PDF/PNG export added."""
    if scatter_kws is None:
        scatter_kws = {}
    if isinstance(figsize, (int, float)):
        figsize = (figsize, figsize)
    scatter_kws_ = {'s': size, 'alpha': 1, 'linewidths': 0}
    scatter_kws_.update(scatter_kws)
    figures = []
    plot_df = df.rename(columns=lambda x: str(x)).copy()
    plot_df[hue] = plot_df[hue].astype('category')
    if invert_y:
        plot_df[Y] = -plot_df[Y]
    sns.set_style({'axes.facecolor': style})
    if exps is None:
        exps = list(plot_df[exp].dropna().unique())
    elif not isinstance(exps, list):
        exps = [exps]
    for name in exps:
        data = plot_df[plot_df[exp] == name]
        if data.empty:
            continue
        (width, height) = figsize
        grid = sns.lmplot(x=X, y=Y, data=data, hue=hue, legend=legend, fit_reg=False, markers='.', height=height, aspect=width / height, palette=palette, scatter=True, scatter_kws=scatter_kws_)
        ax = grid.ax
        ax.set_aspect('equal', adjustable='box')
        ax.set_title(str(name), pad=3)
        if axis == 'off':
            sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)
            ax.set(xticks=[], yticks=[], xlabel='', ylabel='')
        if save_path:
            for fmt in formats:
                out = f'{save_path}_{safe_name(name)}.{fmt}'
                print(f'Saved: {out}')
        plt.show()
        figures.append(grid)
    return figures

def _get_obs_for_catplot():
    needed = ['x', 'y', 'lnp_region', 'condition', 'cell_type', 'cell_type_pooled', 'lnp_positive', 'lnp_call', 'lnp_call_corrected', 'lnp_call_positive_only', 'lnp_detail_formulation']
    adata_backed = ad.read_h5ad(CATPLOT_OUTPUT_H5AD, backed='r')
    available = [col for col in needed if col in adata_backed.obs.columns]
    obs = adata_backed.obs[available].copy()
    adata_backed.file.close()
    if 'x' not in obs.columns or 'y' not in obs.columns:
        raise ValueError("The final h5ad must contain 'x' and 'y' columns in adata.obs.")
    if 'lnp_region' not in obs.columns:
        raise ValueError("The final h5ad must contain 'lnp_region' in adata.obs.")
    obs['lnp_region'] = obs['lnp_region'].astype(str)
    obs = obs[obs['lnp_region'].isin(CATPLOT_LNP_REGIONS)].copy()
    if 'cell_type_pooled' in obs.columns:
        obs['cell_type_display'] = obs['cell_type_pooled'].astype(str)
    elif 'cell_type' in obs.columns:
        obs['cell_type_display'] = obs['cell_type'].astype(str)
    else:
        obs['cell_type_display'] = 'Unknown'
    if 'lnp_call_corrected' in obs.columns:
        obs['lnp_call_display'] = obs['lnp_call_corrected'].astype(str)
    elif 'lnp_call_positive_only' in obs.columns:
        obs['lnp_call_display'] = obs['lnp_call_positive_only'].astype(str)
    elif 'lnp_call' in obs.columns:
        obs['lnp_call_display'] = obs['lnp_call'].astype(str)
    else:
        raise ValueError('The final h5ad must contain an LNP label column.')
    if 'lnp_positive' in obs.columns:
        if pd.api.types.is_bool_dtype(obs['lnp_positive']):
            obs['lnp_positive_bool'] = obs['lnp_positive']
        else:
            obs['lnp_positive_bool'] = obs['lnp_positive'].astype(str).str.lower().isin(['true', '1', 'yes'])
    else:
        obs['lnp_positive_bool'] = obs['lnp_call_display'].isin(CATPLOT_LNP_ORDER)
    obs['x'] = pd.to_numeric(obs['x'], errors='coerce')
    obs['y'] = pd.to_numeric(obs['y'], errors='coerce')
    obs = obs.dropna(subset=['x', 'y'])
    obs['lnp_call_display'] = pd.Categorical(obs['lnp_call_display'], categories=CATPLOT_LNP_ORDER, ordered=True)
    return obs

def _downsample_region_background(region_df, max_cells=CATPLOT_BACKGROUND_MAX_CELLS_PER_REGION, seed=CATPLOT_RANDOM_SEED):
    if max_cells is None or len(region_df) <= max_cells:
        return region_df
    return region_df.sample(n=max_cells, random_state=seed)

def _save_fig(fig, stem, formats=('pdf', 'png')):
    return None

def plot_lnp_spatial_grid(obs, hue, palette, hue_order, title, stem, regions=None, ncols=4, figsize=(7.2, 6.8), background_size=0.06, foreground_size=1.1, background_alpha=0.14, foreground_alpha=0.95, invert_y=True):
    """Overview catplot: gray tissue background plus colored decoded LNP+ cells."""
    if regions is None:
        regions = [reg for reg in CATPLOT_LNP_REGIONS if reg in set(obs['lnp_region'])]
    regions = list(regions)
    nrows = int(np.ceil(len(regions) / ncols))
    (fig, axes) = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    axes_flat = axes.ravel()
    lnp_obs = obs[obs['lnp_positive_bool'] & obs['lnp_call_display'].notna()].copy()
    lnp_obs = lnp_obs[lnp_obs['lnp_call_display'].isin(CATPLOT_LNP_ORDER)]
    for (ax, region) in zip(axes_flat, regions):
        region_all = obs[obs['lnp_region'] == region]
        region_bg = _downsample_region_background(region_all)
        region_lnp = lnp_obs[lnp_obs['lnp_region'] == region]
        bg_y = -region_bg['y'] if invert_y else region_bg['y']
        ax.scatter(region_bg['x'], bg_y, s=background_size, c='#D7D7D7', alpha=background_alpha, linewidths=0, rasterized=True)
        for label in hue_order:
            sub = region_lnp[region_lnp[hue].astype(str) == str(label)]
            if sub.empty:
                continue
            sub_y = -sub['y'] if invert_y else sub['y']
            ax.scatter(sub['x'], sub_y, s=foreground_size, c=palette.get(label, '#333333'), alpha=foreground_alpha, linewidths=0, rasterized=True)
        ax.set_aspect('equal', adjustable='box')
        ax.set_title(region.replace('_', ' '), pad=2)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    for ax in axes_flat[len(regions):]:
        ax.axis('off')
    handles = [plt.Line2D([0], [0], marker='o', linestyle='', markersize=3.4, markerfacecolor=palette.get(label, '#333333'), markeredgecolor='none', label=str(label).replace('_', ' ')) for label in hue_order if str(label) in set(lnp_obs[hue].astype(str))]
    fig.legend(handles=handles, loc='center left', bbox_to_anchor=(0.92, 0.5), frameon=False, title=None, ncol=1, handletextpad=0.5, borderaxespad=0.0)
    fig.suptitle(title, x=0.47, y=0.995, ha='center', va='top', fontsize=9, fontweight='bold')
    fig.subplots_adjust(left=0.01, right=0.88, top=0.94, bottom=0.02, wspace=0.02, hspace=0.08)
    _save_fig(fig, stem)
    plt.show()
    return fig

def plot_lnp_spatial_overlay(obs, hue, palette, hue_order, title_prefix, stem_prefix, regions=None, figsize=(4.0, 4.0), background_size=0.08, foreground_size=1.6, invert_y=True):
    """Individual region plots, useful when a single region needs inspection."""
    if regions is None:
        regions = [reg for reg in CATPLOT_LNP_REGIONS if reg in set(obs['lnp_region'])]
    lnp_obs = obs[obs['lnp_positive_bool'] & obs['lnp_call_display'].notna()].copy()
    lnp_obs = lnp_obs[lnp_obs['lnp_call_display'].isin(CATPLOT_LNP_ORDER)]
    figs = []
    for region in regions:
        region_all = obs[obs['lnp_region'] == region]
        region_bg = _downsample_region_background(region_all)
        region_lnp = lnp_obs[lnp_obs['lnp_region'] == region]
        if region_lnp.empty:
            continue
        (fig, ax) = plt.subplots(figsize=figsize)
        bg_y = -region_bg['y'] if invert_y else region_bg['y']
        ax.scatter(region_bg['x'], bg_y, s=background_size, c='#D7D7D7', alpha=0.16, linewidths=0, rasterized=True)
        for label in hue_order:
            sub = region_lnp[region_lnp[hue].astype(str) == str(label)]
            if sub.empty:
                continue
            sub_y = -sub['y'] if invert_y else sub['y']
            ax.scatter(sub['x'], sub_y, s=foreground_size, c=palette.get(label, '#333333'), alpha=0.95, linewidths=0, rasterized=True, label=str(label).replace('_', ' '))
        ax.set_aspect('equal', adjustable='box')
        ax.set_title(f"{title_prefix}: {region.replace('_', ' ')}", pad=3)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.legend(frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5), markerscale=2.0)
        _save_fig(fig, f'{stem_prefix}_{safe_name(region)}')
        plt.show()
        figs.append(fig)
    return figs


In [ ]:
from pathlib import Path
try:
    catplot_obs
except NameError:
    apply_catplot_nature_style()
    catplot_obs = _get_obs_for_catplot()
SELECTED_REGION_FIGURE_DIR = GENERATED_ROOT / 'Figure_Source_Data'
SELECTED_REGION_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
SELECTED_OVERLAY_REGIONS = ['S2_reg001', 'S2_reg002', 'S2_reg003']
SELECTED_OVERLAY_DPI = 1200

def _collapse_to_b_t_dc_mac_selected(cell_type):
    value = str(cell_type)
    if value == 'B' or value.endswith(' B') or 'B-cell' in value:
        return 'B'
    if value in {'T', 'CD4+ T', 'CD8+ T'} or value.endswith(' T') or 'T-cell' in value:
        return 'T'
    if value == 'DC' or value.endswith(' DC') or 'dendritic' in value.lower():
        return 'DC'
    if value == 'Macrophage' or 'macrophage' in value.lower():
        return 'Macrophage'
    return np.nan
selected_btdc_mac_obs = catplot_obs.copy()
selected_btdc_mac_obs['BTDC_MAC_class'] = selected_btdc_mac_obs['cell_type_display'].map(_collapse_to_b_t_dc_mac_selected)
selected_btdc_mac_obs = selected_btdc_mac_obs[selected_btdc_mac_obs['BTDC_MAC_class'].notna()].copy()
selected_is_lnp01 = selected_btdc_mac_obs['lnp_positive_bool'] & (selected_btdc_mac_obs['lnp_call_display'].astype(str) == 'LNP_01')
selected_is_lnp08 = selected_btdc_mac_obs['lnp_positive_bool'] & (selected_btdc_mac_obs['lnp_call_display'].astype(str) == 'LNP_08')
selected_is_lnp10 = selected_btdc_mac_obs['lnp_positive_bool'] & (selected_btdc_mac_obs['lnp_call_display'].astype(str) == 'LNP_10')
selected_btdc_mac_obs['BTDC_MAC_LNP_overlay'] = 'Other ' + selected_btdc_mac_obs['BTDC_MAC_class'].astype(str)
selected_btdc_mac_obs.loc[selected_is_lnp01, 'BTDC_MAC_LNP_overlay'] = 'LNP 01+ ' + selected_btdc_mac_obs.loc[selected_is_lnp01, 'BTDC_MAC_class'].astype(str)
selected_btdc_mac_obs.loc[selected_is_lnp08, 'BTDC_MAC_LNP_overlay'] = 'LNP 08+ ' + selected_btdc_mac_obs.loc[selected_is_lnp08, 'BTDC_MAC_class'].astype(str)
selected_btdc_mac_obs.loc[selected_is_lnp10, 'BTDC_MAC_LNP_overlay'] = 'LNP 10+ ' + selected_btdc_mac_obs.loc[selected_is_lnp10, 'BTDC_MAC_class'].astype(str)
SELECTED_BTDC_MAC_OVERLAY_ORDER = ['Other B', 'Other T', 'Other DC', 'Other Macrophage', 'LNP 01+ B', 'LNP 01+ T', 'LNP 01+ DC', 'LNP 01+ Macrophage', 'LNP 08+ B', 'LNP 08+ T', 'LNP 08+ DC', 'LNP 08+ Macrophage', 'LNP 10+ B', 'LNP 10+ T', 'LNP 10+ DC', 'LNP 10+ Macrophage']
SELECTED_BTDC_MAC_OVERLAY_PALETTE = {'Other B': '#D9E8F5', 'Other T': '#E5DDF1', 'Other DC': '#D8EBCF', 'Other Macrophage': '#F2E1C9', 'LNP 01+ B': '#1B9E77', 'LNP 01+ T': '#66C2A5', 'LNP 01+ DC': '#006D2C', 'LNP 01+ Macrophage': '#A6D854', 'LNP 08+ B': '#D95F02', 'LNP 08+ T': '#E7298A', 'LNP 08+ DC': '#B2182B', 'LNP 08+ Macrophage': '#F6A01A', 'LNP 10+ B': '#386CB0', 'LNP 10+ T': '#00BFC4', 'LNP 10+ DC': '#542788', 'LNP 10+ Macrophage': '#7B68EE'}

def _save_selected_region_fig(fig, stem):
    return None

def plot_selected_btdc_mac_region(region, figsize=(5.2, 5.2), invert_y=True):
    region_all = catplot_obs[catplot_obs['lnp_region'] == region]
    region_bg = _downsample_region_background(region_all, max_cells=None)
    region_overlay = selected_btdc_mac_obs[selected_btdc_mac_obs['lnp_region'] == region]
    if region_all.empty:
        raise ValueError(f'No cells found for {region}. Check the region name.')
    (fig, ax) = plt.subplots(figsize=figsize)
    bg_y = -region_bg['y'] if invert_y else region_bg['y']
    ax.scatter(region_bg['x'], bg_y, s=0.08, c='#D9D9D9', alpha=0.12, linewidths=0, rasterized=True)
    for label in ['Other B', 'Other T', 'Other DC', 'Other Macrophage']:
        sub = region_overlay[region_overlay['BTDC_MAC_LNP_overlay'] == label]
        if sub.empty:
            continue
        sub_y = -sub['y'] if invert_y else sub['y']
        ax.scatter(sub['x'], sub_y, s=0.32, c=SELECTED_BTDC_MAC_OVERLAY_PALETTE[label], alpha=0.42, linewidths=0, rasterized=True)
    for label in ['LNP 01+ B', 'LNP 01+ T', 'LNP 01+ DC', 'LNP 01+ Macrophage', 'LNP 08+ B', 'LNP 08+ T', 'LNP 08+ DC', 'LNP 08+ Macrophage', 'LNP 10+ B', 'LNP 10+ T', 'LNP 10+ DC', 'LNP 10+ Macrophage']:
        sub = region_overlay[region_overlay['BTDC_MAC_LNP_overlay'] == label]
        if sub.empty:
            continue
        sub_y = -sub['y'] if invert_y else sub['y']
        ax.scatter(sub['x'], sub_y, s=1.65, c=SELECTED_BTDC_MAC_OVERLAY_PALETTE[label], alpha=0.98, linewidths=0, rasterized=True)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(region.replace('_', ' '), pad=3, fontsize=9, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    present_labels = [label for label in SELECTED_BTDC_MAC_OVERLAY_ORDER if label in set(region_overlay['BTDC_MAC_LNP_overlay'])]
    handles = [plt.Line2D([0], [0], marker='o', linestyle='', markersize=4.2, markerfacecolor=SELECTED_BTDC_MAC_OVERLAY_PALETTE[label], markeredgecolor='none', alpha=0.95 if label.startswith('LNP') else 0.55, label=label) for label in present_labels]
    ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, handletextpad=0.4, borderaxespad=0.0)
    _save_selected_region_fig(fig, f'selected_region_lnp01_lnp08_lnp10_btdc_mac_overlay_{safe_name(region)}')
    plt.show()
    return fig
selected_region_counts = selected_btdc_mac_obs[selected_btdc_mac_obs['lnp_region'].isin(SELECTED_OVERLAY_REGIONS)].groupby(['lnp_region', 'BTDC_MAC_class', 'BTDC_MAC_LNP_overlay'], observed=True).size().reset_index(name='n_cells')
selected_counts_path = SELECTED_REGION_FIGURE_DIR / 'selected_region_lnp01_lnp08_lnp10_btdc_mac_overlay_counts.csv'
selected_region_counts.to_csv(selected_counts_path, index=False)
print(f'Saved source table: {selected_counts_path}')
for selected_region in SELECTED_OVERLAY_REGIONS:
    plot_selected_btdc_mac_region(selected_region, figsize=(7.4, 7.4))


## 10. Final manuscript plotting code

These cells were copied from the later figure-generation notebooks used to assemble the manuscript. They read the tables produced above and display the final plots inline.


In [ ]:
CELL_TABLE_DIR = TABLE_DIR


### LNP-only per-LNP cellular uptake boxplot

Nature-style LNP-only boxplot of per-LNP cellular uptake. Colors encode region (`reg000`-`reg003`).

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = CELL_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
BY_REGION_CSV = TABLE_DIR / 'lnp_uptake_percent_by_region.csv'
PDF_PATH = OUT / 'lnp_only_per_lnp_cellular_uptake_box_nature.pdf'
PNG_PATH = OUT / 'lnp_only_per_lnp_cellular_uptake_box_nature.png'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.65, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.65, 'ytick.major.width': 0.65, 'xtick.major.size': 2.4, 'ytick.major.size': 2.4, 'legend.frameon': False, 'savefig.dpi': 600})
uptake = pd.read_csv(BY_REGION_CSV)
lnp_order = [f'LNP_{i:02d}' for i in range(1, 11)]
plot_df = uptake[(uptake['condition'] == 'LNP') & uptake['lnp_call'].isin(lnp_order)].copy()
plot_df['lnp_call'] = pd.Categorical(plot_df['lnp_call'], categories=lnp_order, ordered=True)
plot_df = plot_df.sort_values(['lnp_call', 'lnp_region'])
plot_df['region_id'] = plot_df['lnp_region'].str.extract('_(reg\\d+)$')[0]
region_order = sorted(plot_df['region_id'].dropna().unique(), key=lambda x: int(x.replace('reg', '')))
region_palette = {'reg000': '#2C7FB8', 'reg001': '#41AB5D', 'reg002': '#F28E2B', 'reg003': '#8E6BBE'}
replicate_labels = {'reg000': 'Bio. rep. 1', 'reg001': 'Bio. rep. 2', 'reg002': 'Bio. rep. 3', 'reg003': 'Bio. rep. 4'}
neutral = '#222222'
grid = '#E7E7E7'
box_fill = '#F5F7FA'
label_map = dict(zip(lnp_order, [f'LNP {i:02d}' for i in range(1, 11)]))
x = np.arange(len(lnp_order))
(fig, ax) = plt.subplots(figsize=(4.6, 2.75))
box_data = [plot_df.loc[plot_df['lnp_call'].astype(str) == lnp, 'pct_cells_lnp_positive'].dropna().to_numpy(float) for lnp in lnp_order]
box = ax.boxplot(box_data, positions=x, widths=0.56, patch_artist=True, showfliers=False, medianprops={'color': neutral, 'linewidth': 0.9}, whiskerprops={'color': neutral, 'linewidth': 0.7}, capprops={'color': neutral, 'linewidth': 0.7}, boxprops={'edgecolor': neutral, 'linewidth': 0.7})
for patch in box['boxes']:
    patch.set_facecolor(box_fill)
    patch.set_alpha(0.98)
rng = np.random.default_rng(21)
for (i, lnp) in enumerate(lnp_order):
    sub = plot_df[plot_df['lnp_call'].astype(str) == lnp].copy()
    jitter = rng.uniform(-0.17, 0.17, len(sub))
    for (k, (_, row)) in enumerate(sub.iterrows()):
        ax.scatter(i + jitter[k], row['pct_cells_lnp_positive'], s=10, marker='o', facecolor=region_palette.get(row['region_id'], '#999999'), edgecolor='white', linewidth=0.35, alpha=0.82, zorder=3)
ax.set_title('Per-LNP cellular uptake in LNP-treated regions', fontsize=8.7, fontweight='bold', pad=5)
ax.set_ylabel('Cells LNP+ (%)')
ax.set_xlabel('')
ax.set_xticks(x)
ax.set_xticklabels([label_map[v] for v in lnp_order], rotation=45, ha='right')
ymax = float(np.nanmax(plot_df['pct_cells_lnp_positive'].to_numpy(float)))
ax.set_ylim(0, math.ceil(ymax * 1.15 * 10) / 10)
ax.set_xlim(-0.65, len(lnp_order) - 0.35)
ax.tick_params(axis='x', pad=1)
region_handles = [Line2D([0], [0], marker='o', linestyle='none', markersize=4.2, markerfacecolor=region_palette[r], markeredgecolor='white', markeredgewidth=0.4, label=replicate_labels.get(r, r)) for r in region_order]
ax.legend(handles=region_handles, loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=6.2, title='Biological replicate', title_fontsize=6.4, borderaxespad=0, handletextpad=0.5)
fig.text(0.46, 0.012, 'Dots show individual slide measurements; colors denote biological replicates.', ha='center', va='bottom', fontsize=6.1, color='#555555')
fig.subplots_adjust(left=0.105, right=0.77, bottom=0.3, top=0.86)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
plot_df.to_csv(OUT / 'lnp_only_per_lnp_cellular_uptake_box_source_data.csv', index=False)
print(f"n LNP region measurements per LNP: {plot_df.groupby('lnp_call', observed=True)['lnp_region'].nunique().to_dict()}")


### LNP+ cell-type composition

Nature-style 100% stacked bar replot of cell-type composition among LNP+ cells. Percent labels are shown for readable segments.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = CELL_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
COMPOSITION_CSV = TABLE_DIR / 'lnp_positive_cell_type_composition.csv'
PDF_PATH = OUT / 'lnp_positive_cell_type_composition_stacked_nature.pdf'
PNG_PATH = OUT / 'lnp_positive_cell_type_composition_stacked_nature.png'
SOURCE_PATH = OUT / 'lnp_positive_cell_type_composition_stacked_source_data.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.65, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.65, 'ytick.major.width': 0.65, 'xtick.major.size': 2.4, 'ytick.major.size': 2.4, 'legend.frameon': False, 'savefig.dpi': 600})
comp = pd.read_csv(COMPOSITION_CSV)
lnp_order = [f'LNP_{i:02d}' for i in range(1, 11)]
cell_order = ['B', 'Macrophage', 'Endothelial', 'CD4+ T', 'CD8+ T', 'DC', 'Fibroblast', 'Muscle', 'Neutrophil', 'Stromal']
cell_colors = {'B': '#2C7FB8', 'Macrophage': '#8DD783', 'Endothelial': '#F28E2B', 'CD4+ T': '#6BAED6', 'DC': '#2CA02C', 'CD8+ T': '#9ECAE1', 'Fibroblast': '#D62728', 'Muscle': '#FF9896', 'Neutrophil': '#9467BD', 'Stromal': '#C5B0D5'}
label_map = dict(zip(lnp_order, [f'LNP {i:02d}' for i in range(1, 11)]))
plot = comp[comp['lnp_call'].isin(lnp_order) & comp['cell_type'].isin(cell_order)].copy()
plot['lnp_call'] = pd.Categorical(plot['lnp_call'], categories=lnp_order, ordered=True)
plot['cell_type'] = pd.Categorical(plot['cell_type'], categories=cell_order, ordered=True)
plot = plot.sort_values(['lnp_call', 'cell_type'])
wide = plot.pivot(index='lnp_call', columns='cell_type', values='pct_of_lnp_positive_cells').reindex(index=lnp_order, columns=cell_order).fillna(0)
counts = plot.drop_duplicates('lnp_call').set_index('lnp_call').reindex(lnp_order)['n_lnp_positive_total'].astype(int)
source = wide.reset_index().melt(id_vars='lnp_call', var_name='cell_type', value_name='pct_of_lnp_positive_cells')
source = source.merge(plot[['lnp_call', 'cell_type', 'n_lnp_positive', 'n_lnp_positive_total', 'formulation']].astype({'lnp_call': str, 'cell_type': str}), on=['lnp_call', 'cell_type'], how='left')
source.to_csv(SOURCE_PATH, index=False)
(fig, ax) = plt.subplots(figsize=(4.95, 3.2))
y = np.arange(len(lnp_order))
left = np.zeros(len(lnp_order), dtype=float)
for cell_type in cell_order:
    vals = wide[cell_type].to_numpy(float)
    bars = ax.barh(y, vals, left=left, height=0.7, color=cell_colors[cell_type], edgecolor='white', linewidth=0.45, label=cell_type, zorder=2)
    for (i, v) in enumerate(vals):
        if v >= 4.0:
            (r, g, b) = mpl.colors.to_rgb(cell_colors[cell_type])
            luminance = 0.299 * r + 0.587 * g + 0.114 * b
            txt_color = 'white' if luminance < 0.55 else '#111111'
            fs = 5.4 if v < 6 else 5.8
            ax.text(left[i] + v / 2, y[i], f'{v:.0f}%', ha='center', va='center', fontsize=fs, color=txt_color, fontweight='bold' if v >= 8 else 'normal', clip_on=True)
    left += vals
ax.set_yticks(y)
ax.set_yticklabels([label_map[v] for v in lnp_order])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel('Cell type among LNP+ cells (%)')
ax.set_ylabel('')
ax.set_title('Cell-type composition among LNP+ cells', fontsize=9.2, fontweight='bold', pad=6)
ax.set_xticks([0, 25, 50, 75, 100])
ax.tick_params(axis='y', length=0)
for (i, lnp) in enumerate(lnp_order):
    ax.text(101.2, y[i], f'n={counts.loc[lnp]:,}', ha='left', va='center', fontsize=5.8, color='#555555')
ax.text(101.2, -0.78, 'LNP+ cells', ha='left', va='center', fontsize=5.8, color='#555555')
handles = [Patch(facecolor=cell_colors[ct], edgecolor='none', label=ct) for ct in cell_order]
ax.legend(handles=handles, title='Cell type', title_fontsize=6.5, fontsize=6.2, loc='center left', bbox_to_anchor=(1.2, 0.5), borderaxespad=0, handlelength=1.0, handleheight=0.8, labelspacing=0.45)
fig.text(0.08, 0.018, 'Percent labels are shown for cell-type fractions >=4%.', ha='left', va='bottom', fontsize=5.8, color='#555555')
fig.subplots_adjust(left=0.13, right=0.7, bottom=0.17, top=0.9)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')


### OVA+ LNP+ selected cell-type LNP-label composition

Nature-style dot-matrix replot. Values are normalized within each selected cell type across decoded LNP labels, excluding no-LNP-signal cells.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = CELL_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'luc_ova_positive_selected_cell_type_lnp_label_composition.csv'
PDF_PATH = OUT / 'ova_positive_lnp_label_composition_dotmatrix_nature.pdf'
PNG_PATH = OUT / 'ova_positive_lnp_label_composition_dotmatrix_nature.png'
SOURCE_PATH = OUT / 'ova_positive_lnp_label_composition_dotmatrix_source_data.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.65, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.65, 'ytick.major.width': 0.65, 'xtick.major.size': 2.4, 'ytick.major.size': 2.4, 'legend.frameon': False, 'savefig.dpi': 600})
lnp_order = [f'LNP_{i:02d}' for i in range(1, 11)]
cell_order = ['DC', 'CD8+ T', 'CD4+ T', 'B', 'Macrophage']
cell_labels = {'DC': 'DC', 'CD8+ T': 'CD8+ T', 'CD4+ T': 'CD4+ T', 'B': 'B', 'Macrophage': 'Macrophage'}
lnp_labels = dict(zip(lnp_order, [f'LNP {i:02d}' for i in range(1, 11)]))
raw = pd.read_csv(INPUT_CSV)
plot = raw[(raw['marker'] == 'OVA') & raw['cell_type'].isin(cell_order) & raw['lnp_label_for_gene_positive'].isin(lnp_order)].copy()
plot = plot.rename(columns={'lnp_label_for_gene_positive': 'lnp_call'})
plot['decoded_lnp_positive_total_in_cell_type'] = plot.groupby('cell_type', observed=True)['n_positive_cells'].transform('sum')
plot['pct_among_ova_positive_lnp_positive'] = np.where(plot['decoded_lnp_positive_total_in_cell_type'] > 0, 100 * plot['n_positive_cells'] / plot['decoded_lnp_positive_total_in_cell_type'], np.nan)
plot['cell_type'] = pd.Categorical(plot['cell_type'], categories=cell_order, ordered=True)
plot['lnp_call'] = pd.Categorical(plot['lnp_call'], categories=lnp_order, ordered=True)
plot = plot.sort_values(['cell_type', 'lnp_call'])
plot.to_csv(SOURCE_PATH, index=False)
matrix = plot.pivot(index='lnp_call', columns='cell_type', values='pct_among_ova_positive_lnp_positive').reindex(index=lnp_order, columns=cell_order)
vals = matrix.to_numpy(dtype=float)
vmax = float(np.nanmax(vals))
color_max = max(30, 5 * np.ceil(vmax / 5))
cmap = LinearSegmentedColormap.from_list('ova_lnp', ['#F7FBFF', '#C6DBEF', '#6BAED6', '#2171B5', '#08306B'])
norm = Normalize(vmin=0, vmax=color_max)
(fig, ax) = plt.subplots(figsize=(4.9, 3.05))
for yi in range(len(lnp_order)):
    if yi % 2 == 0:
        ax.axhspan(yi - 0.5, yi + 0.5, color='#F8F8F8', zorder=0)
for (xi, cell_type) in enumerate(cell_order):
    for (yi, lnp) in enumerate(lnp_order):
        pct = matrix.loc[lnp, cell_type]
        if not np.isfinite(pct):
            continue
        size = 18 + pct / color_max * 270
        ax.scatter(xi, yi, s=size, color=cmap(norm(pct)), edgecolor='white', linewidth=0.55, zorder=2)
        label = f'{pct:.0f}%' if pct >= 10 else f'{pct:.1f}%'
        txt_color = 'white' if pct >= color_max * 0.48 else '#1a1a1a'
        ax.text(xi, yi, label, ha='center', va='center', fontsize=5.7, color=txt_color, fontweight='bold' if pct >= 12 else 'normal', zorder=3)
ax.set_xticks(np.arange(len(cell_order)))
ax.set_xticklabels([cell_labels[c] for c in cell_order], rotation=0, ha='center')
ax.xaxis.tick_top()
ax.tick_params(axis='x', pad=5, length=0)
ax.set_yticks(np.arange(len(lnp_order)))
ax.set_yticklabels([lnp_labels[l] for l in lnp_order])
ax.tick_params(axis='y', length=0)
ax.invert_yaxis()
ax.set_xlim(-0.55, len(cell_order) - 0.45)
ax.set_ylim(len(lnp_order) - 0.5, -0.5)
ax.set_title('LNP labels among OVA+ LNP+ selected cell types', fontsize=9.2, fontweight='bold', pad=25)
ax.set_xlabel('')
ax.set_ylabel('')
for spine in ax.spines.values():
    spine.set_visible(False)
guide_vals = [5, 15, 30]
handles = []
for g in guide_vals:
    handles.append(ax.scatter([], [], s=18 + g / color_max * 270, color=cmap(norm(g)), edgecolor='white', linewidth=0.55, label=f'{g}%'))
leg = ax.legend(handles=handles, title='Composition', title_fontsize=6.5, fontsize=6.2, loc='center left', bbox_to_anchor=(1.04, 0.62), borderaxespad=0, labelspacing=0.9, handletextpad=0.8)
ax.add_artist(leg)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cax = fig.add_axes([0.82, 0.22, 0.018, 0.36])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('% of OVA+ LNP+ cells', fontsize=6.2, labelpad=4)
cbar.ax.tick_params(labelsize=5.8, length=2, width=0.5)
cbar.outline.set_linewidth(0.4)
fig.text(0.1, 0.026, 'Percentages are normalized within each cell type across decoded LNP labels.', ha='left', va='bottom', fontsize=5.9, color='#555555')
fig.subplots_adjust(left=0.14, right=0.76, bottom=0.15, top=0.8)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')


### SIINFEKL-H-2Kb+ LNP+ DC LNP-label composition

Nature-style horizontal bar replot. Only total n is shown; individual per-LNP n values are omitted.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = CELL_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'siinfekl_positive_lnp_dc_lnp_composition.csv'
PDF_PATH = OUT / 'siinfekl_positive_lnp_dc_lnp_composition_bar_nature.pdf'
PNG_PATH = OUT / 'siinfekl_positive_lnp_dc_lnp_composition_bar_nature.png'
SOURCE_PATH = OUT / 'siinfekl_positive_lnp_dc_lnp_composition_bar_source_data.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.65, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.65, 'ytick.major.width': 0.65, 'xtick.major.size': 2.4, 'ytick.major.size': 2.4, 'legend.frameon': False, 'savefig.dpi': 600})
lnp_order = [f'LNP_{i:02d}' for i in range(1, 11)]
lnp_labels = dict(zip(lnp_order, [f'LNP {i:02d}' for i in range(1, 11)]))
lnp_colors = {'LNP_01': '#1F77B4', 'LNP_02': '#AEC7E8', 'LNP_03': '#FF7F0E', 'LNP_04': '#FFBB78', 'LNP_05': '#2CA02C', 'LNP_06': '#98DF8A', 'LNP_07': '#D62728', 'LNP_08': '#FF9896', 'LNP_09': '#9467BD', 'LNP_10': '#C5B0D5'}
plot = pd.read_csv(INPUT_CSV)
plot = plot[plot['lnp_call'].isin(lnp_order)].copy()
plot['lnp_call'] = pd.Categorical(plot['lnp_call'], categories=lnp_order, ordered=True)
plot = plot.sort_values('lnp_call')
plot['display_label'] = plot['lnp_call'].astype(str).map(lnp_labels)
plot.to_csv(SOURCE_PATH, index=False)
total_n = int(plot['n_siinfekl_positive_lnp_dc'].sum())
(fig, ax) = plt.subplots(figsize=(3.85, 2.8))
y = np.arange(len(plot))
vals = plot['pct_siinfekl_positive_lnp_dc'].to_numpy(float)
colors = [lnp_colors[str(x)] for x in plot['lnp_call'].astype(str)]
ax.barh(y, vals, height=0.62, color=colors, edgecolor='#222222', linewidth=0.45, zorder=2)
for (yi, v) in zip(y, vals):
    ax.text(v + 0.45, yi, f'{v:.1f}%', ha='left', va='center', fontsize=6.4, color='#222222')
ax.set_yticks(y)
ax.set_yticklabels(plot['display_label'])
ax.invert_yaxis()
ax.set_xlabel('Composition of SIINFEKL-H-2Kb+ LNP+ DCs (%)')
ax.set_ylabel('')
ax.set_xlim(0, max(vals) * 1.25)
ax.set_title(f'LNP composition of SIINFEKL-H-2Kb+ LNP+ DCs\nTotal n={total_n:,}', fontsize=8.8, fontweight='bold', pad=7)
ax.tick_params(axis='y', length=0)
class_handles = [Patch(facecolor='#D9D9D9', edgecolor='none', label='SM-102: LNP 01-05'), Patch(facecolor='#8C8C8C', edgecolor='none', label='ALC-0315: LNP 06-10')]
ax.text(0.99, -0.22, 'LNP 01-05: SM-102; LNP 06-10: ALC-0315', transform=ax.transAxes, ha='right', va='top', fontsize=5.8, color='#555555')
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.22, top=0.82)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')


### B-cell LNP uptake versus OVA+ readout

Compares LNP-label composition among all LNP+ B cells with LNP-label composition among OVA+ LNP+ B cells.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = CELL_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
UPTAKE_CSV = TABLE_DIR / 'lnp_positive_cell_type_composition.csv'
TRANSFECTION_CSV = TABLE_DIR / 'luc_ova_positive_selected_cell_type_lnp_label_composition.csv'
PDF_PATH = OUT / 'b_cell_uptake_vs_ova_transfection_comparison_nature.pdf'
PNG_PATH = OUT / 'b_cell_uptake_vs_ova_transfection_comparison_nature.png'
SOURCE_PATH = OUT / 'b_cell_uptake_vs_ova_transfection_comparison_source_data.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
uptake = pd.read_csv(UPTAKE_CSV)
uptake_b = uptake[uptake['cell_type'].eq('B')].copy()
uptake_b = uptake_b[['lnp_call', 'n_lnp_positive', 'lnp_number', 'formulation']].copy()
uptake_b['uptake_composition_pct'] = uptake_b['n_lnp_positive'] / uptake_b['n_lnp_positive'].sum() * 100
trans = pd.read_csv(TRANSFECTION_CSV)
trans_b = trans[trans['cell_type'].eq('B') & trans['marker'].eq('OVA') & trans['lnp_label_for_gene_positive'].str.startswith('LNP_', na=False)].copy()
trans_b = trans_b.rename(columns={'lnp_label_for_gene_positive': 'lnp_call', 'n_positive_cells': 'n_ova_positive_lnp_b'})
trans_b = trans_b[['lnp_call', 'n_ova_positive_lnp_b']].copy()
trans_b['ova_transfection_composition_pct'] = trans_b['n_ova_positive_lnp_b'] / trans_b['n_ova_positive_lnp_b'].sum() * 100
df = uptake_b.merge(trans_b, on='lnp_call', how='inner')
df['composition_delta_transfection_minus_uptake'] = df['ova_transfection_composition_pct'] - df['uptake_composition_pct']
df['lnp_label'] = df['lnp_call'].str.replace('_', ' ', regex=False)
df = df.sort_values('ova_transfection_composition_pct', ascending=True).reset_index(drop=True)
(rho, spearman_p) = stats.spearmanr(df['uptake_composition_pct'], df['ova_transfection_composition_pct'])
(pearson_r, pearson_p) = stats.pearsonr(df['uptake_composition_pct'], df['ova_transfection_composition_pct'])
df['spearman_rho'] = rho
df['spearman_pvalue'] = spearman_p
df['pearson_r'] = pearson_r
df['pearson_pvalue'] = pearson_p
df.to_csv(SOURCE_PATH, index=False)
uptake_color = '#B8C96B'
trans_color = '#C65A4A'
line_color = '#C8C8C8'
(fig, ax) = plt.subplots(figsize=(3.25, 2.3))
y = np.arange(len(df))
for (i, row) in df.iterrows():
    x_u = row['uptake_composition_pct']
    x_t = row['ova_transfection_composition_pct']
    ax.hlines(i, min(x_u, x_t), max(x_u, x_t), color=line_color, linewidth=0.75, zorder=1)
    ax.scatter(x_u, i, s=22, color=uptake_color, edgecolor='#333333', linewidth=0.35, zorder=3)
    ax.scatter(x_t, i, s=22, color=trans_color, edgecolor='white', linewidth=0.35, zorder=4)
    if row['lnp_call'] in {'LNP_05', 'LNP_10'}:
        ax.text(max(x_u, x_t) + 0.55, i, f"{row['composition_delta_transfection_minus_uptake']:+.1f}", ha='left', va='center', fontsize=5.8, color='#333333')
ax.set_yticks(y)
ax.set_yticklabels(df['lnp_label'], fontsize=6.2)
ax.set_xlabel('Composition among B-cell population (%)')
ax.set_xlim(0, max(df['uptake_composition_pct'].max(), df['ova_transfection_composition_pct'].max()) + 6.8)
ax.set_ylim(-0.6, len(df) - 0.4)
ax.set_title('B-cell LNP uptake versus OVA+ readout', fontsize=8.4, fontweight='bold', pad=5)
ax.grid(False)
ax.tick_params(axis='y', length=0, pad=2)
ax.tick_params(axis='x', labelsize=6.2)
handles = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=uptake_color, markeredgecolor='#333333', markeredgewidth=0.35, markersize=4.8, label='LNP+ B cells'), Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=trans_color, markeredgecolor='white', markeredgewidth=0.35, markersize=4.8, label='OVA+ LNP+ B cells')]
ax.legend(handles=handles, loc='lower right', bbox_to_anchor=(0.98, 0.05), fontsize=6.1, handletextpad=0.45, labelspacing=0.35)
fig.text(0.22, 0.035, f'Spearman rho={rho:.2f}', ha='left', va='bottom', fontsize=5.8, color='#555555')
fig.text(0.96, 0.035, 'Values at right show OVA+ - uptake', ha='right', va='bottom', fontsize=5.6, color='#777777')
fig.subplots_adjust(left=0.2, right=0.96, bottom=0.25, top=0.8)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f"Total LNP+ B cells: {int(df['n_lnp_positive'].sum()):,}")
print(f"Total OVA+ LNP+ B cells: {int(df['n_ova_positive_lnp_b'].sum()):,}")
print(f'Spearman rho={rho:.3f}, p={spearman_p:.4g}; Pearson r={pearson_r:.3f}, p={pearson_p:.4g}')
print(df[['lnp_call', 'n_lnp_positive', 'uptake_composition_pct', 'n_ova_positive_lnp_b', 'ova_transfection_composition_pct', 'composition_delta_transfection_minus_uptake']].sort_values('ova_transfection_composition_pct', ascending=False).to_string(index=False))


### OVA+ LNP+ selected cell-type LNP-label pie charts

Nature-style pie-chart replot. Values are normalized within each selected cell type across decoded LNP labels, excluding no-LNP-signal cells.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = CELL_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'luc_ova_positive_selected_cell_type_lnp_label_composition.csv'
PDF_PATH = OUT / 'ova_positive_lnp_label_composition_pies_nature.pdf'
PNG_PATH = OUT / 'ova_positive_lnp_label_composition_pies_nature.png'
SOURCE_PATH = OUT / 'ova_positive_lnp_label_composition_pies_source_data.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.65, 'legend.frameon': False, 'savefig.dpi': 600})
lnp_order = [f'LNP_{i:02d}' for i in range(1, 11)]
cell_order = ['DC', 'CD8+ T', 'CD4+ T', 'B', 'Macrophage']
lnp_labels = dict(zip(lnp_order, [f'LNP {i:02d}' for i in range(1, 11)]))
lnp_colors = {'LNP_01': '#1F77B4', 'LNP_02': '#AEC7E8', 'LNP_03': '#FF7F0E', 'LNP_04': '#FFBB78', 'LNP_05': '#2CA02C', 'LNP_06': '#98DF8A', 'LNP_07': '#D62728', 'LNP_08': '#FF9896', 'LNP_09': '#9467BD', 'LNP_10': '#C5B0D5'}
raw = pd.read_csv(INPUT_CSV)
plot = raw[(raw['marker'] == 'OVA') & raw['cell_type'].isin(cell_order) & raw['lnp_label_for_gene_positive'].isin(lnp_order)].copy()
plot = plot.rename(columns={'lnp_label_for_gene_positive': 'lnp_call'})
plot['decoded_lnp_positive_total_in_cell_type'] = plot.groupby('cell_type', observed=True)['n_positive_cells'].transform('sum')
plot['pct_among_ova_positive_lnp_positive'] = np.where(plot['decoded_lnp_positive_total_in_cell_type'] > 0, 100 * plot['n_positive_cells'] / plot['decoded_lnp_positive_total_in_cell_type'], np.nan)
plot.to_csv(SOURCE_PATH, index=False)
wide = plot.pivot(index='cell_type', columns='lnp_call', values='pct_among_ova_positive_lnp_positive').reindex(index=cell_order, columns=lnp_order).fillna(0)
(fig, axes) = plt.subplots(1, 5, figsize=(7.15, 2.25), subplot_kw={'aspect': 'equal'})
fig.patch.set_facecolor('white')
for (ax, cell_type) in zip(axes, cell_order):
    vals = wide.loc[cell_type, lnp_order].to_numpy(float)
    colors = [lnp_colors[l] for l in lnp_order]
    (wedges, _) = ax.pie(vals, colors=colors, startangle=90, counterclock=False, radius=0.88, labels=None, wedgeprops={'linewidth': 0.55, 'edgecolor': 'white'})
    for (wedge, pct) in zip(wedges, vals):
        if pct < 4:
            continue
        theta = np.deg2rad((wedge.theta1 + wedge.theta2) / 2)
        (r, g, b) = mpl.colors.to_rgb(wedge.get_facecolor())[:3]
        lum = 0.299 * r + 0.587 * g + 0.114 * b
        label = f'{pct:.0f}%' if pct >= 8 else f'{pct:.1f}%'
        if pct >= 12:
            (x, y) = (0.54 * np.cos(theta), 0.54 * np.sin(theta))
            ax.text(x, y, label, ha='center', va='center', fontsize=5.7, color='white' if lum < 0.55 else '#111111', fontweight='bold')
        else:
            (x0, y0) = (0.86 * np.cos(theta), 0.86 * np.sin(theta))
            (x1, y1) = (1.08 * np.cos(theta), 1.08 * np.sin(theta))
            ha = 'left' if x1 >= 0 else 'right'
            ax.annotate(label, xy=(x0, y0), xytext=(x1, y1), ha=ha, va='center', fontsize=5.3, color='#222222', arrowprops={'arrowstyle': '-', 'color': '#777777', 'linewidth': 0.35, 'shrinkA': 0, 'shrinkB': 0})
    ax.set_title(cell_type, fontsize=8.2, fontweight='bold', pad=5)
    ax.set_xlim(-1.28, 1.28)
    ax.set_ylim(-1.18, 1.18)
handles = [Line2D([0], [0], marker='o', linestyle='none', markersize=5.0, markerfacecolor=lnp_colors[l], markeredgecolor='none', label=lnp_labels[l]) for l in lnp_order]
fig.legend(handles=handles, title='LNP label', title_fontsize=6.6, fontsize=6.2, loc='center left', bbox_to_anchor=(0.895, 0.5), borderaxespad=0, handletextpad=0.6, labelspacing=0.42)
fig.suptitle('LNP labels among OVA+ LNP+ selected cell types', fontsize=9.4, fontweight='bold', y=0.99)
fig.text(0.06, 0.02, 'Percentages are normalized within each cell type across decoded LNP labels; slices <4% are unlabeled.', ha='left', va='bottom', fontsize=5.8, color='#555555')
fig.subplots_adjust(left=0.035, right=0.865, bottom=0.2, top=0.78, wspace=0.3)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
